# 08 — Shared Sequence Loader + Leakage-Safe Standardisation

**Goal.** Build one trusted data-access layer (`bsl_sequence_loader.py`) that notebooks 09–13 import unchanged, so every run reads the same bytes through the same code. It trains nothing.

**What it freezes**
- **Folds and paths:** fold membership comes from 07 and is rebuilt and verified, never regenerated. `relative_path` is authoritative.
- **Length policy:** `MAX_FRAMES = 256`. All frames are kept if T ≤ 256, otherwise 256 evenly spaced indices are taken over the whole clip (72 clips affected). No interpolation, repetition or truncation.
- **Padding:** dynamic per batch, applied *after* standardisation, with true `lengths` and a temporal mask kept separate from the 4 presence masks.
- **Standardisation:** fold scalers are summed from the 06 per-signer statistics over the **12 training signers only**. The control scaler is streamed over the 4,500 control-training clips only. Presence-mask columns are never z-scored.
- **Scope-safe batching:** the public batch functions fetch the correct standardiser internally and take no `standardiser` argument, so another fold's or branch's scaling cannot be substituted by mistake.
- **Integrity:** the module's SHA-256 is recorded, and downstream notebooks hash it *before* importing.

**Requirement passed to the neural notebooks.** Sequences must go through `pack_padded_sequence(..., enforce_sorted=False)` (or equivalent masking). Otherwise a bidirectional RNN's backward pass starts on padding. Temporal pooling must divide by the true lengths.

**Deliberately not frozen here.** Optimiser, architecture and RF settings belong to 10–12. The only fairness rule carried here is that within one model, Full and Reduced go through an identical procedure.

**Inputs → Outputs.** 06, 07, 05a, 05b → `bsl_sequence_loader.py`, `loader_manifest.json`, `fold_standardisers_{b}.npz`, `control_standardiser_{b}.npz`, `loader_validation_report.{json,csv}`.

In [ ]:
# ============================================================
# CELL 1 — Configuration
# ============================================================
from __future__ import annotations

import importlib
import json
import os
import platform
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

NOTEBOOK = "nb8"
LOADER_VERSION = "nb8-v1"
RANDOM_SEED = 42

# Provenance this notebook refuses to proceed without. Every value was printed by
# a real Kaggle run of notebook 6 or notebook 7.
REQUIRED_EXPERIMENT_ID = "1308155924042fc2"     # notebook 7 final
REQUIRED_MANIFEST_VERSION = "nb7-v5"
REQUIRED_PIPELINE_ID = "24c9d5ccb3f1"           # phase 5, both branches
REQUIRED_CLIP_TABLE_HASH = "73273a14e599f85c"   # notebook 6 canonical clip table
REQUIRED_SUCCESSFUL_CLIP_ID_HASH = "e9295d6f4e66e25b"   # notebook 6 successful clips
REQUIRED_HASHES = {
    "manifest": "0bc51c943ed2e688",
    "fold_assignments": "bc94d5ba486ff49c",
    "signer_pairs": "3bd636fe92328543",
    "pair_order": "c5fe49f66f772507",
    "control_split": "fcc419d92e546470",
    "label_map": "83ac9fcbf7c01a59",
}

EXPECTED = {
    "clips": 6000, "labels": 200, "signers": 16, "folds": 8,
    "train_signers": 12, "validation_signers": 2, "test_signers": 2,
    "control_train_clips": 4500, "total_frames": 806159,
    "widths": {"full": 1759, "reduced": 631},
    "face_end": {"full": 1404, "reduced": 276},
    "mask_slice": {"full": (1755, 1759), "reduced": (627, 631)},
    "shared_tail_width": 355, "max_frames": 256,
}

SEARCH_ROOT = "/kaggle/input"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

MODULE_PATH = WORKING / "bsl_sequence_loader.py"
LOADER_MANIFEST = WORKING / "loader_manifest.json"
REPORT_JSON = WORKING / "loader_validation_report.json"
REPORT_CSV = WORKING / "loader_validation_report.csv"
FOLD_STD_PATH = {b: WORKING / f"fold_standardisers_{b}.npz" for b in ("full", "reduced")}
CONTROL_STD_PATH = {b: WORKING / f"control_standardiser_{b}.npz" for b in ("full", "reduced")}

STREAM_WORKERS = min(4, os.cpu_count() or 1)

ENVIRONMENT = {
    "notebook": NOTEBOOK, "loader_version": LOADER_VERSION,
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "seed": RANDOM_SEED,
    "cpu_count": os.cpu_count(), "stream_workers": STREAM_WORKERS,
}

STARTED = time.time()
CHECKS: List[Dict[str, Any]] = []


def record(name: str, passed: bool, detail: Any = "", status: str = "") -> bool:
    """Log a check and keep it for loader_validation_report.*"""
    passed = bool(passed)
    status = status or ("pass" if passed else "fail")
    CHECKS.append({"check": name, "passed": passed, "status": status,
                   "detail": str(detail)})
    label = {"pass": "PASS", "fail": "FAIL", "skip": "SKIP"}[status]
    print(f"  {label}  {name:<58} {detail}")
    return passed


def skip(name: str, reason: Any) -> None:
    """Record a check that could not run here. Never counted as a pass."""
    record(name, True, reason, status="skip")


def failures() -> List[Dict[str, Any]]:
    return [entry for entry in CHECKS if entry["status"] == "fail"]


def require(name: str, passed: bool, detail: Any = "") -> None:
    """A check that must hold. Fail loudly rather than write a trusted-looking file."""
    if not record(name, passed, detail):
        raise RuntimeError(f"{name} failed ({detail}). Notebook 8 will not publish a "
                           "loader contract on top of a failed check.")


print("Environment")
for key, value in ENVIRONMENT.items():
    print(f"  {key}: {value}")

print(f"\nRequired provenance")
print(f"  experiment id           : {REQUIRED_EXPERIMENT_ID}")
print(f"  manifest version        : {REQUIRED_MANIFEST_VERSION}")
print(f"  pipeline id             : {REQUIRED_PIPELINE_ID}")
print(f"  clip table hash         : {REQUIRED_CLIP_TABLE_HASH}")
print(f"  successful clip hash    : {REQUIRED_SUCCESSFUL_CLIP_ID_HASH}")
for name, digest in REQUIRED_HASHES.items():
    print(f"  {name:<24}: {digest}")

print(f"\nDesign: {EXPECTED['folds']} folds | "
      f"{EXPECTED['train_signers']} train / {EXPECTED['validation_signers']} validation / "
      f"{EXPECTED['test_signers']} test signers")
print(f"Branches: full {EXPECTED['widths']['full']} | reduced {EXPECTED['widths']['reduced']} "
      f"features per frame, {EXPECTED['shared_tail_width']} shared non-face columns")
print(f"Neural policy: keep every frame up to {EXPECTED['max_frames']}, "
      f"uniform reduction above it, dynamic batch padding")
print("\nThis notebook trains nothing and freezes no model hyperparameter.")

In [ ]:
# ============================================================
# CELL 2 — Write bsl_sequence_loader.py, then import the artefact itself
# ============================================================
# The module is written to disk first and every later cell exercises the IMPORTED
# copy. That way the self-tests validate the exact file notebooks 9 to 13 will
# import, not a parallel in-notebook definition that could drift from it.

MODULE_SOURCE = r"""
'''bsl_sequence_loader - shared data-access layer for the frozen BSL experiment.

Written by notebook 8. Imported unchanged by notebooks 9 to 13.

This module freezes the DATA and LOADER contract only. It contains no optimiser,
no learning rate, no epoch budget, no architecture and no Random Forest setting.
Those belong to the individual model notebooks and must be frozen there before
any test evaluation. The only fairness requirement this module imposes is that,
within a given model, the full and reduced branches are driven through an
identical procedure. BiLSTM and BiGRU need not match each other.

Experiment id 1308155924042fc2 - refuse to run against anything else.
'''
from __future__ import annotations

import hashlib
import json
import os
import re
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

# ==========================================================================
# 1. The frozen contract. Every value below is verified, never inferred.
# ==========================================================================
LOADER_VERSION = 'nb8-v1'

REQUIRED_EXPERIMENT_ID = '1308155924042fc2'
REQUIRED_MANIFEST_VERSION = 'nb7-v5'
REQUIRED_PIPELINE_ID = '24c9d5ccb3f1'
REQUIRED_CLIP_TABLE_HASH = '73273a14e599f85c'
REQUIRED_SUCCESSFUL_CLIP_ID_HASH = 'e9295d6f4e66e25b'
REQUIRED_AUDIT_VERDICT = 'PASS'

REQUIRED_HASHES: Dict[str, str] = {
    'manifest': '0bc51c943ed2e688',
    'fold_assignments': 'bc94d5ba486ff49c',
    'signer_pairs': '3bd636fe92328543',
    'pair_order': 'c5fe49f66f772507',
    'control_split': 'fcc419d92e546470',
    'label_map': '83ac9fcbf7c01a59',
}

# Notebook 7 cell 7: sha256 of the '|'-joined component values, first 16 hex.
EXPERIMENT_ID_COMPONENTS = ('clip_table', 'manifest', 'fold_assignments',
                            'signer_pairs', 'pair_order', 'control_split',
                            'label_map', 'manifest_version')

BRANCHES: Tuple[str, ...] = ('full', 'reduced')
BRANCH_WIDTH: Dict[str, int] = {'full': 1759, 'reduced': 631}
FACE_END: Dict[str, int] = {'full': 1404, 'reduced': 276}       # = tail_start
MASK_SLICE: Dict[str, Tuple[int, int]] = {'full': (1755, 1759),
                                          'reduced': (627, 631)}
SHARED_TAIL_WIDTH = 355          # width - face_end, identical in both branches

N_FOLDS = 8
N_SIGNERS = 16
N_CLIPS = 6000
N_LABELS = 200
TRAIN_SIGNERS_PER_FOLD = 12
VALIDATION_SIGNERS_PER_FOLD = 2
TEST_SIGNERS_PER_FOLD = 2
CONTROL_TRAIN_CLIPS = 4500

# Neural sequence policy. Nothing here is a hyperparameter to tune.
MAX_FRAMES = 256
PAD_VALUE = 0.0
STD_FLOOR = 1e-8

MANIFEST_HASH_COLUMNS = ['clip_id', 'label', 'label_id', 'signer_id', 'session',
                         'relative_path', 'original_frames']
FOLD_HASH_COLUMNS = ['fold', 'clip_id', 'split']
CONTROL_HASH_COLUMNS = ['clip_id', 'split']

MANIFEST_TEXT_COLUMNS = ['clip_id', 'label', 'signer_id', 'session', 'relative_path']
FOLD_TEXT_COLUMNS = ['clip_id', 'signer_id', 'split']
CONTROL_TEXT_COLUMNS = ['clip_id', 'signer_id', 'label', 'split', 'split_type']

SPLITS = ('train', 'validation', 'test')


class ProvenanceError(RuntimeError):
    '''Raised when the attached data is not this experiment.'''


class LeakageError(RuntimeError):
    '''Raised when a transformation would be fitted on non-training signers.'''


class ScopeError(RuntimeError):
    '''Raised when data from one scope is paired with another scope's standardiser.

    Fold 0's clips scaled by fold 1's standardiser, or control clips scaled by a
    fold standardiser, is preprocessing leakage that no later check would catch.
    The safe batching API refuses it structurally rather than trusting callers.
    '''


# Batch fields that must never be moved off the CPU. pack_padded_sequence
# requires a CPU lengths tensor whatever device the model sits on.
DEVICE_EXEMPT_KEYS: Tuple[str, ...] = ('lengths',)

# Keys that must not appear anywhere inside the scientific loader contract, so
# that rerunning the same loader tomorrow reproduces the same contract id.
VOLATILE_CONTRACT_KEYS = frozenset({
    'generated_at', 'generated_at_utc', 'timestamp', 'runtime_seconds',
    'environment', 'resolved_paths', 'resolved_paths_this_session',
    'csv_read_strategy', 'notebook7_dir', 'cpu_count', 'python', 'numpy',
    'pandas', 'stream_workers', 'hostname', 'run_metadata', 'module_sha256',
    'sha256', 'session', 'paths', 'standardiser_digests', 'digest',
})
_TIMESTAMP_PATTERN = re.compile(r'\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}')


# ==========================================================================
# 2. Hashing - byte-for-byte the notebook 7 recipe
# ==========================================================================
def sha16(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]


def canonical_hash(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    '''Notebook 7 cell 7 verbatim: mergesort on the columns, tab-joined
    'column=value' lines, trailing newline, sha256 truncated to 16 hex.'''
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise ProvenanceError('frame is missing hash columns: ' + repr(missing))
    ordered = frame.sort_values(list(columns), kind='mergesort')[list(columns)]
    lines = ['\t'.join(f'{column}={record[column]}' for column in columns)
             for record in ordered.to_dict('records')]
    return hashlib.sha256(('\n'.join(lines) + '\n').encode('utf-8')).hexdigest()[:16]


def json_hash(payload: Any) -> str:
    return sha16(json.dumps(payload, separators=(',', ':')))


def compute_experiment_id(components: Dict[str, Any]) -> str:
    missing = [k for k in EXPERIMENT_ID_COMPONENTS if components.get(k) is None]
    if missing:
        raise ProvenanceError('cannot recompute the experiment id, missing: '
                              + repr(missing))
    joined = '|'.join(str(components[name]) for name in EXPERIMENT_ID_COMPONENTS)
    return hashlib.sha256(joined.encode('utf-8')).hexdigest()[:16]


def array_digest(*arrays: np.ndarray) -> str:
    digest = hashlib.sha256()
    for array in arrays:
        contiguous = np.ascontiguousarray(array)
        digest.update(str(contiguous.dtype).encode('utf-8'))
        digest.update(str(contiguous.shape).encode('utf-8'))
        digest.update(contiguous.tobytes())
    return digest.hexdigest()[:16]


# ==========================================================================
# 3. Discovery. Absolute paths from notebook 7 are stale by design.
# ==========================================================================
def find_all(pattern: str, search_root: str = '/kaggle/input') -> List[Path]:
    root = Path(search_root)
    if not root.exists():
        raise FileNotFoundError(f'search root {root} does not exist')
    return sorted(p for p in root.glob(pattern))


def find_one(pattern: str, description: str,
             search_root: str = '/kaggle/input') -> Path:
    hits = find_all(pattern, search_root)
    if not hits:
        raise FileNotFoundError(
            f'No {description} found under {search_root} (pattern {pattern!r}). '
            'Attach the producing notebook output.')
    if len(hits) > 1:
        listing = '\n'.join(f'    {h}' for h in hits)
        raise ProvenanceError(
            f'{len(hits)} candidates for {description}; refusing to choose.\n{listing}')
    return hits[0]


def normalise_relative(value: str) -> str:
    '''Notebook 6 stored posix relative paths. Defend against separators anyway.'''
    return PurePosixPath(str(value).replace('\\', '/')).as_posix()


# ==========================================================================
# 4. Frozen table reading with hash verification
# ==========================================================================
def read_frozen_csv(path: Path, hash_columns: Sequence[str], expected_hash: str,
                    text_columns: Sequence[str]) -> Tuple[pd.DataFrame, str, List[Tuple[str, str]]]:
    '''Read a frozen csv and require it to reproduce its notebook 7 hash.

    Two read strategies are tried. Both parse the same bytes; they differ only
    in how pandas coerces text columns, which changes the formatted value the
    hash is computed over. The hash still has to match exactly, so this cannot
    launder a corrupted file - it only prevents a csv-typing convention from
    masquerading as a provenance failure. The strategy that matched is recorded.
    '''
    attempts: List[Tuple[str, str]] = []
    strategies = (
        ('text-preserving', {'dtype': {c: str for c in text_columns},
                             'keep_default_na': False}),
        ('pandas-default', {}),
    )
    for name, kwargs in strategies:
        try:
            frame = pd.read_csv(path, **kwargs)
        except Exception as error:                       # unreadable under this strategy
            attempts.append((name, f'read failed: {type(error).__name__}'))
            continue
        digest = canonical_hash(frame, hash_columns)
        attempts.append((name, digest))
        if digest == expected_hash:
            return frame, name, attempts
    detail = '; '.join(f'{n} -> {d}' for n, d in attempts)
    raise ProvenanceError(
        f'{path.name} does not reproduce its frozen hash {expected_hash}. '
        f'Tried: {detail}. This file is not the notebook 7 artefact.')


# ==========================================================================
# 5. Per-signer sufficient statistics published by notebook 6
# ==========================================================================
@dataclass(frozen=True)
class SignerStatistics:
    branch: str
    width: int
    tail_start: int
    mask_columns: np.ndarray
    signers: Tuple[str, ...]
    frames: Dict[str, int]
    sums: Dict[str, np.ndarray]
    sumsq: Dict[str, np.ndarray]
    total_frames: int
    provenance: Dict[str, Any]


def load_signer_statistics(path: Path, branch: str) -> SignerStatistics:
    with np.load(path, allow_pickle=False) as archive:

        def meta(name: str) -> Any:
            key = f'__{name}__'
            if key not in archive.files:
                raise ProvenanceError(
                    f'{path.name} lacks {key}; it was written by an older audit.')
            return archive[key][0]

        provenance = {
            'branch': str(meta('branch')),
            'audit_verdict': str(meta('audit_verdict')),
            'audit_version': str(meta('audit_version')),
            'pipeline_id': str(meta('pipeline_id')),
            'clip_table_hash': str(meta('clip_table_hash')),
            'successful_clip_id_hash': str(meta('successful_clip_id_hash')),
            'successful_clip_count': int(meta('successful_clip_count')),
            'width': int(meta('width')),
            'tail_start': int(meta('tail_start')),
            'total_frames': int(meta('total_frames')),
        }
        expected = {
            'branch': branch,
            'audit_verdict': REQUIRED_AUDIT_VERDICT,
            'pipeline_id': REQUIRED_PIPELINE_ID,
            'clip_table_hash': REQUIRED_CLIP_TABLE_HASH,
            'successful_clip_id_hash': REQUIRED_SUCCESSFUL_CLIP_ID_HASH,
            'successful_clip_count': N_CLIPS,
            'width': BRANCH_WIDTH[branch],
            'tail_start': FACE_END[branch],
        }
        wrong = {k: (provenance[k], v) for k, v in expected.items()
                 if provenance[k] != v}
        if wrong:
            detail = '; '.join(f'{k}: found {f!r}, required {e!r}'
                               for k, (f, e) in wrong.items())
            raise ProvenanceError(
                f'{path.name} does not belong to this experiment. {detail}')

        mask_columns = np.asarray(archive['__mask_columns__'], dtype=np.int64)
        expected_mask = np.arange(*MASK_SLICE[branch], dtype=np.int64)
        if not np.array_equal(mask_columns, expected_mask):
            raise ProvenanceError(
                f'{branch} mask columns are {mask_columns.tolist()}, '
                f'required {expected_mask.tolist()}')

        signers = tuple(str(s) for s in archive['__signers__'])
        if len(signers) != N_SIGNERS or len(set(signers)) != N_SIGNERS:
            raise ProvenanceError(f'{path.name} holds {len(signers)} signers, '
                                  f'required {N_SIGNERS}')

        frames: Dict[str, int] = {}
        sums: Dict[str, np.ndarray] = {}
        sumsq: Dict[str, np.ndarray] = {}
        for signer in signers:
            frames[signer] = int(archive[f'{signer}|frames'][0])
            signer_sum = np.asarray(archive[f'{signer}|sum'], dtype=np.float64)
            signer_sumsq = np.asarray(archive[f'{signer}|sumsq'], dtype=np.float64)
            if signer_sum.shape != (BRANCH_WIDTH[branch],) or \
               signer_sumsq.shape != (BRANCH_WIDTH[branch],):
                raise ProvenanceError(
                    f'{signer} statistics have shape {signer_sum.shape}, '
                    f'required ({BRANCH_WIDTH[branch]},)')
            if not (np.isfinite(signer_sum).all() and np.isfinite(signer_sumsq).all()):
                raise ProvenanceError(f'{signer} statistics contain NaN or Inf')
            sums[signer] = signer_sum
            sumsq[signer] = signer_sumsq

    total = sum(frames.values())
    if total != provenance['total_frames']:
        raise ProvenanceError(
            f'{path.name}: per-signer frames sum to {total:,} but __total_frames__ '
            f'is {provenance["total_frames"]:,}')

    return SignerStatistics(
        branch=branch, width=BRANCH_WIDTH[branch], tail_start=FACE_END[branch],
        mask_columns=mask_columns, signers=signers, frames=frames, sums=sums,
        sumsq=sumsq, total_frames=total, provenance=provenance)


# ==========================================================================
# 6. Standardisation - the notebook 6 mathematics, written exactly once
# ==========================================================================
@dataclass
class Standardiser:
    '''Frame-level z-scoring for one branch and one fitting scope.

    mean     = sum(x) / frames
    variance = sum(x^2) / frames - mean^2
    std      = sqrt(max(variance, 0))
    std[std < 1e-8] = 1
    mean[mask columns] = 0, std[mask columns] = 1   (presence masks stay binary)

    This covers frame-level features only. Derived Random Forest features live
    in a different space and need their own training-only scaler, fitted in the
    model notebook - never reuse this one for them.
    '''
    branch: str
    scope: str
    mean: np.ndarray
    std: np.ndarray
    n_frames: int
    fitted_on: Dict[str, Any] = field(default_factory=dict)
    # scope_kind is what the safe batching API checks. 'ad_hoc' is anything built
    # for a probe or a self-test and is refused by make_fold_batch /
    # make_control_batch by design.
    scope_kind: str = 'ad_hoc'
    fold: Optional[int] = None

    def __post_init__(self) -> None:
        _require_branch(self.branch)
        width = BRANCH_WIDTH[self.branch]
        self.mean = np.asarray(self.mean, dtype=np.float64).copy()
        self.std = np.asarray(self.std, dtype=np.float64).copy()
        if self.mean.shape != (width,) or self.std.shape != (width,):
            raise ValueError(f'{self.branch} standardiser must be ({width},), got '
                             f'{self.mean.shape} and {self.std.shape}')
        if not (np.isfinite(self.mean).all() and np.isfinite(self.std).all()):
            raise ValueError('standardiser contains NaN or Inf')
        if (self.std <= 0).any():
            raise ValueError('standardiser has a non-positive std')
        mask = self.mask_columns
        if not (np.all(self.mean[mask] == 0.0) and np.all(self.std[mask] == 1.0)):
            raise ValueError('presence-mask columns must have mean 0 and std 1')
        self._mean32 = self.mean.astype(np.float32)
        self._std32 = self.std.astype(np.float32)

    @property
    def mask_columns(self) -> np.ndarray:
        return np.arange(*MASK_SLICE[self.branch], dtype=np.int64)

    @property
    def tail_start(self) -> int:
        return FACE_END[self.branch]

    def apply(self, sequence: np.ndarray) -> np.ndarray:
        '''(T, D) -> (T, D) float32. Mask columns pass through untouched
        because their offset is exactly 0.0 and their scale exactly 1.0.'''
        array = np.asarray(sequence)
        if array.ndim != 2 or array.shape[1] != self.mean.shape[0]:
            raise ValueError(f'expected (T, {self.mean.shape[0]}), got {array.shape}')
        return ((array.astype(np.float32, copy=False) - self._mean32)
                / self._std32).astype(np.float32, copy=False)

    def apply_float64(self, sequence: np.ndarray) -> np.ndarray:
        '''Reference implementation used only by the self-tests.'''
        return (np.asarray(sequence, dtype=np.float64) - self.mean) / self.std

    def digest(self) -> str:
        return array_digest(self.mean, self.std)

    def summary(self) -> Dict[str, Any]:
        return {'branch': self.branch, 'scope': self.scope,
                'scope_kind': self.scope_kind,
                'fold': None if self.fold is None else int(self.fold),
                'n_frames': int(self.n_frames), 'digest': self.digest(),
                'mean_min': float(self.mean.min()), 'mean_max': float(self.mean.max()),
                'std_min': float(self.std.min()), 'std_max': float(self.std.max()),
                'floored_columns': int((self.std == 1.0).sum()),
                'fitted_on': self.fitted_on}


def standardiser_from_totals(branch: str, frames: int, total_sum: np.ndarray,
                             total_sumsq: np.ndarray, scope: str,
                             fitted_on: Optional[Dict[str, Any]] = None,
                             scope_kind: str = 'ad_hoc',
                             fold: Optional[int] = None) -> Standardiser:
    '''The single place the standardisation mathematics is implemented.'''
    if frames <= 0:
        raise ValueError(f'{scope}: {frames} frames, cannot standardise')
    width = BRANCH_WIDTH[branch]
    total_sum = np.asarray(total_sum, dtype=np.float64)
    total_sumsq = np.asarray(total_sumsq, dtype=np.float64)
    if total_sum.shape != (width,) or total_sumsq.shape != (width,):
        raise ValueError(f'{scope}: totals must be ({width},)')

    mean = total_sum / frames
    variance = total_sumsq / frames - mean ** 2
    std = np.sqrt(np.maximum(variance, 0.0))
    std = np.where(std < STD_FLOOR, 1.0, std)
    mask = np.arange(*MASK_SLICE[branch], dtype=np.int64)
    mean[mask] = 0.0
    std[mask] = 1.0
    if scope_kind not in ('fold', 'control', 'ad_hoc'):
        raise ValueError(f'unknown scope_kind {scope_kind!r}')
    if (scope_kind == 'fold') != (fold is not None):
        raise ValueError("scope_kind 'fold' requires a fold number, and only it may "
                         'carry one')
    return Standardiser(branch=branch, scope=scope, mean=mean, std=std,
                        n_frames=int(frames), fitted_on=dict(fitted_on or {}),
                        scope_kind=scope_kind, fold=fold)


def sum_signer_statistics(statistics: SignerStatistics,
                          signers: Sequence[str]) -> Tuple[int, np.ndarray, np.ndarray]:
    '''Add per-signer sufficient statistics in a fixed order, so the float64
    accumulation is bit-reproducible across runs.'''
    ordered = sorted(set(signers), key=_signer_number)
    if len(ordered) != len(list(signers)):
        raise ValueError('duplicate signers passed to sum_signer_statistics')
    unknown = [s for s in ordered if s not in statistics.frames]
    if unknown:
        raise ValueError(f'unknown signers {unknown} for branch {statistics.branch}')
    frames = 0
    total_sum = np.zeros(statistics.width, dtype=np.float64)
    total_sumsq = np.zeros(statistics.width, dtype=np.float64)
    for signer in ordered:
        frames += statistics.frames[signer]
        total_sum += statistics.sums[signer]
        total_sumsq += statistics.sumsq[signer]
    return frames, total_sum, total_sumsq


def _signer_number(signer: str) -> int:
    return int(str(signer).lstrip('p'))


# ==========================================================================
# 7. Neural sequence policy
# ==========================================================================
def uniform_indices(n_frames: int, max_frames: int = MAX_FRAMES) -> np.ndarray:
    '''Every frame when T <= max_frames; otherwise max_frames evenly spaced
    indices over the WHOLE sequence, keeping the first and last frame.

    No stretching, no repetition, no interpolation, no tail truncation, no
    first-N window. Deterministic: no random state is consulted.
    '''
    n_frames = int(n_frames)
    if n_frames <= 0:
        raise ValueError(f'{n_frames} frames')
    if n_frames <= max_frames:
        return np.arange(n_frames, dtype=np.int64)
    indices = np.rint(np.linspace(0.0, n_frames - 1.0, max_frames)).astype(np.int64)
    if indices[0] != 0 or indices[-1] != n_frames - 1:
        raise AssertionError('uniform sampling dropped the first or last frame')
    if not np.all(np.diff(indices) > 0):
        raise AssertionError('uniform sampling produced a repeated frame index')
    return indices


def collate_variable_length(sequences: Sequence[np.ndarray],
                            label_ids: Sequence[int],
                            clip_ids: Sequence[str],
                            signer_ids: Optional[Sequence[str]] = None,
                            scope: Optional[str] = None) -> Dict[str, Any]:
    '''Dynamic padding to the batch maximum - never to MAX_FRAMES.

    Returns
      sequence      (B, Lmax, D) float32, padded with 0.0 AFTER standardisation
      lengths       (B,)         int64, true frame counts
      temporal_mask (B, Lmax)    bool, True = real frame, False = padding
      label_id      (B,)         int64
      clip_id, signer_id        lists

    temporal_mask is NOT the MediaPipe presence mask. The four presence columns
    say whether face/pose/left/right hand were detected in a real frame; this
    says whether the frame exists at all.
    '''
    if len(sequences) == 0:
        raise ValueError('empty batch')
    if not (len(sequences) == len(label_ids) == len(clip_ids)):
        raise ValueError('batch component lengths disagree')
    widths = {int(s.shape[1]) for s in sequences}
    if len(widths) != 1:
        raise ValueError(f'mixed feature widths in one batch: {sorted(widths)}')
    width = widths.pop()

    lengths = np.array([int(s.shape[0]) for s in sequences], dtype=np.int64)
    longest = int(lengths.max())
    batch = np.full((len(sequences), longest, width), PAD_VALUE, dtype=np.float32)
    temporal_mask = np.zeros((len(sequences), longest), dtype=bool)
    for row, sequence in enumerate(sequences):
        length = int(sequence.shape[0])
        batch[row, :length] = sequence.astype(np.float32, copy=False)
        temporal_mask[row, :length] = True

    if int(temporal_mask.sum()) != int(lengths.sum()):
        raise AssertionError('temporal mask disagrees with the true lengths')
    return {'sequence': batch, 'lengths': lengths, 'temporal_mask': temporal_mask,
            'label_id': np.asarray(label_ids, dtype=np.int64),
            'clip_id': list(clip_ids),
            'signer_id': list(signer_ids) if signer_ids is not None else None,
            'width': width, 'padded_length': longest, 'scope': scope}


def to_torch(batch: Dict[str, Any], device: Any = None) -> Dict[str, Any]:
    '''Optional bridge. Imported lazily so this module never needs torch.

    `lengths` is deliberately left on the CPU whatever device is requested.
    torch.nn.utils.rnn.pack_padded_sequence takes its lengths argument on the
    CPU; handing it a CUDA tensor either raises or forces a silent copy back.
    Everything the model consumes -- sequence, temporal_mask, label_id -- moves
    to the requested device. DEVICE_EXEMPT_KEYS records the exemption.

    Neural notebooks MUST feed sequence + lengths through
    pack_padded_sequence(..., enforce_sorted=False), or a demonstrably equivalent
    masking implementation. A bidirectional recurrence fed the padded tensor
    directly starts its backward pass on padding and carries that state into the
    real frames.
    '''
    import torch

    movable = {
        'sequence': torch.from_numpy(np.ascontiguousarray(batch['sequence'])),
        'temporal_mask': torch.from_numpy(np.ascontiguousarray(batch['temporal_mask'])),
        'label_id': torch.from_numpy(np.ascontiguousarray(batch['label_id'])),
    }
    if device is not None:
        movable = {key: tensor.to(device) for key, tensor in movable.items()}

    out: Dict[str, Any] = dict(movable)
    # Built after the move, from numpy, so it cannot inherit the device.
    out['lengths'] = torch.from_numpy(
        np.ascontiguousarray(batch['lengths'], dtype=np.int64))
    if out['lengths'].device.type != 'cpu':
        raise AssertionError('lengths must stay on the CPU for pack_padded_sequence')
    if out['lengths'].dtype != torch.int64:
        raise AssertionError('lengths must be int64 for pack_padded_sequence')

    out['clip_id'] = batch['clip_id']
    out['signer_id'] = batch['signer_id']
    out['device_exempt'] = list(DEVICE_EXEMPT_KEYS)
    out['scope'] = batch.get('scope')
    return out


def epoch_order(clip_ids: Sequence[str], seed: int, epoch: int) -> List[str]:
    '''Deterministic per-epoch permutation. Same (seed, epoch) -> same order,
    on any machine, so all runs of a branch see identical batch composition.'''
    generator = np.random.default_rng([int(seed), int(epoch)])
    ordered = sorted(clip_ids)
    permutation = generator.permutation(len(ordered))
    return [ordered[i] for i in permutation]


# ==========================================================================
# 8. The loader context
# ==========================================================================
@dataclass
class LoaderContext:
    search_root: str
    branch_roots: Dict[str, Path]
    schemas: Dict[str, Dict[str, Any]]
    statistics_paths: Dict[str, Path]
    manifest: pd.DataFrame
    fold_assignments: pd.DataFrame
    control_assignments: pd.DataFrame
    label_to_id: Dict[str, int]
    id_to_label: Dict[int, str]
    folds: Dict[int, Dict[str, Tuple[str, ...]]]
    signer_pairs: List[List[str]]
    pair_order: List[int]
    provenance: Dict[str, Any]
    _stats: Dict[str, SignerStatistics] = field(default_factory=dict)
    _fold_standardisers: Dict[Tuple[int, str], Standardiser] = field(default_factory=dict)
    _control_standardisers: Dict[str, Standardiser] = field(default_factory=dict)
    _row: Dict[str, int] = field(default_factory=dict)

    # ---- clip metadata ---------------------------------------------------
    def __post_init__(self) -> None:
        self._row = {clip: index for index, clip
                     in enumerate(self.manifest['clip_id'].tolist())}

    def clip_row(self, clip_id: str) -> pd.Series:
        if clip_id not in self._row:
            raise KeyError(f'{clip_id} is not in the frozen manifest')
        return self.manifest.iloc[self._row[clip_id]]

    def frames_for(self, clip_id: str) -> int:
        return int(self.clip_row(clip_id)['original_frames'])

    def label_id_for(self, clip_id: str) -> int:
        return int(self.clip_row(clip_id)['label_id'])

    def signer_for(self, clip_id: str) -> str:
        return str(self.clip_row(clip_id)['signer_id'])

    def path_for(self, clip_id: str, branch: str) -> Path:
        _require_branch(branch)
        relative = normalise_relative(self.clip_row(clip_id)['relative_path'])
        return self.branch_roots[branch] / relative

    # ---- split membership -----------------------------------------------
    def clip_ids(self, fold: int, split: str) -> List[str]:
        if fold not in self.folds:
            raise KeyError(f'fold {fold} does not exist')
        if split not in SPLITS:
            raise KeyError(f'{split!r} is not one of {SPLITS}')
        rows = self.fold_assignments
        selected = rows[(rows['fold'] == fold) & (rows['split'] == split)]
        return sorted(selected['clip_id'].astype(str).tolist())

    def control_clip_ids(self, split: str) -> List[str]:
        if split not in SPLITS:
            raise KeyError(f'{split!r} is not one of {SPLITS}')
        rows = self.control_assignments
        return sorted(rows.loc[rows['split'] == split, 'clip_id'].astype(str).tolist())

    def train_signers(self, fold: int) -> Tuple[str, ...]:
        return self.folds[fold]['train']

    # ---- raw loading (the Random Forest path) ---------------------------
    def load_raw_sequence(self, clip_id: str, branch: str,
                          validate: bool = True) -> np.ndarray:
        '''Original variable-length sequence, (T, D) float32.

        No 256-frame reduction, no padding, no standardisation. This is what
        the Random Forest path consumes; feature engineering there must operate
        over real frames only.
        '''
        _require_branch(branch)
        path = self.path_for(clip_id, branch)
        if not path.exists():
            raise FileNotFoundError(f'{clip_id} ({branch}) missing at {path}')
        array = np.load(path, allow_pickle=False)
        if not validate:
            return np.ascontiguousarray(array, dtype=np.float32)

        width = BRANCH_WIDTH[branch]
        if array.ndim != 2 or int(array.shape[1]) != width:
            raise ValueError(f'{clip_id} ({branch}) has shape {array.shape}, '
                             f'expected (T, {width})')
        expected_frames = self.frames_for(clip_id)
        if int(array.shape[0]) != expected_frames:
            raise ValueError(f'{clip_id} ({branch}) holds {array.shape[0]} frames, '
                             f'the manifest says {expected_frames}')
        if not np.isfinite(array).all():
            raise ValueError(f'{clip_id} ({branch}) contains NaN or Inf')
        start, end = MASK_SLICE[branch]
        masks = array[:, start:end]
        if not np.isin(masks, (0.0, 1.0)).all():
            raise ValueError(f'{clip_id} ({branch}) presence mask is not binary')
        if Path(path).stem != str(clip_id):
            raise ValueError(f'{clip_id} resolved to {path.name}; manifest and disk disagree')
        return np.ascontiguousarray(array, dtype=np.float32)

    # ---- standardisers ---------------------------------------------------
    def signer_statistics(self, branch: str) -> SignerStatistics:
        _require_branch(branch)
        if branch not in self._stats:
            self._stats[branch] = load_signer_statistics(
                self.statistics_paths[branch], branch)
        return self._stats[branch]

    def validate_training_signers(self, fold: int, signers: Sequence[str]) -> Tuple[str, ...]:
        '''Refuse to fit on anything but the 12 training signers of the fold.'''
        definition = self.folds[fold]
        requested = tuple(sorted(set(signers), key=_signer_number))
        if len(requested) != len(list(signers)):
            raise LeakageError(f'fold {fold}: duplicate signers requested')
        held_out = set(definition['validation']) | set(definition['test'])
        intruders = sorted(set(requested) & held_out, key=_signer_number)
        if intruders:
            raise LeakageError(
                f'fold {fold}: {intruders} are validation or test signers and must '
                'never contribute to the standardiser.')
        if requested != tuple(sorted(definition['train'], key=_signer_number)):
            raise LeakageError(
                f'fold {fold}: expected exactly the {TRAIN_SIGNERS_PER_FOLD} training '
                f'signers {list(definition["train"])}, got {list(requested)}')
        return requested

    def fold_standardiser(self, fold: int, branch: str,
                          signers: Optional[Sequence[str]] = None) -> Standardiser:
        _require_branch(branch)
        if fold not in self.folds:
            raise KeyError(f'fold {fold} does not exist')
        requested = self.folds[fold]['train'] if signers is None else signers
        training = self.validate_training_signers(fold, requested)
        key = (fold, branch)
        if key in self._fold_standardisers and signers is None:
            return self._fold_standardisers[key]
        statistics = self.signer_statistics(branch)
        frames, total_sum, total_sumsq = sum_signer_statistics(statistics, training)
        standardiser = standardiser_from_totals(
            branch, frames, total_sum, total_sumsq, scope=f'fold:{fold}',
            scope_kind='fold', fold=int(fold),
            fitted_on={'kind': 'primary_fold', 'fold': int(fold),
                       'training_signers': list(training),
                       'excluded_validation': list(self.folds[fold]['validation']),
                       'excluded_test': list(self.folds[fold]['test']),
                       'source': 'notebook 6 per-signer sufficient statistics'})
        if signers is None:
            self._fold_standardisers[key] = standardiser
        return standardiser

    def register_control_standardiser(self, standardiser: Standardiser) -> None:
        if standardiser.scope_kind != 'control':
            raise ScopeError(f'refusing to register a {standardiser.scope_kind!r} '
                             'standardiser as the control standardiser')
        self._control_standardisers[standardiser.branch] = standardiser

    def control_standardiser(self, branch: str,
                             path: Optional[Path] = None) -> Standardiser:
        _require_branch(branch)
        if branch in self._control_standardisers:
            return self._control_standardisers[branch]
        if path is None:
            path = find_one(f'**/control_standardiser_{branch}.npz',
                            f'control standardiser ({branch})', self.search_root)
        loaded = load_standardisers(Path(path))
        if 'control' not in loaded:
            raise ProvenanceError(f'{Path(path).name} has no "control" entry')
        self._control_standardisers[branch] = loaded['control']
        return self._control_standardisers[branch]

    # ---- the neural path -------------------------------------------------
    def neural_sequence(self, clip_id: str, branch: str,
                        standardiser: Standardiser) -> np.ndarray:
        '''Raw -> uniform reduction only if T > 256 -> standardisation.

        Padding happens per batch, not here. Reducing before scaling is
        arithmetically identical to scaling before reducing and touches fewer
        rows.
        '''
        if standardiser.branch != branch:
            raise ValueError(f'standardiser is for {standardiser.branch}, not {branch}')
        raw = self.load_raw_sequence(clip_id, branch)
        indices = uniform_indices(raw.shape[0])
        reduced = raw if indices.shape[0] == raw.shape[0] else raw[indices]
        return standardiser.apply(reduced)

    # ---- unguarded internals. Private on purpose: they accept ANY standardiser
    # ---- for ANY clips, which is exactly the mistake the public API prevents.
    # ---- Notebooks 9 to 13 must not call these.
    def _make_batch(self, clip_ids: Sequence[str], branch: str,
                    standardiser: Standardiser,
                    scope: Optional[str] = None) -> Dict[str, Any]:
        sequences = [self.neural_sequence(c, branch, standardiser) for c in clip_ids]
        return collate_variable_length(
            sequences,
            [self.label_id_for(c) for c in clip_ids],
            list(clip_ids),
            [self.signer_for(c) for c in clip_ids],
            scope=scope)

    def _iterate_batches(self, clip_ids: Sequence[str], branch: str,
                         standardiser: Standardiser, batch_size: int,
                         shuffle: bool = False, seed: int = 42, epoch: int = 0):
        order = epoch_order(clip_ids, seed, epoch) if shuffle else sorted(clip_ids)
        for start in range(0, len(order), batch_size):
            yield self._make_batch(order[start:start + batch_size], branch, standardiser)

    # ---- scope-safe API: this is what notebooks 10 to 12 should call --------
    def require_standardiser_scope(self, standardiser: Standardiser, branch: str,
                                   scope_kind: str,
                                   fold: Optional[int] = None) -> Standardiser:
        '''Refuse any standardiser that was not fitted for exactly this scope.'''
        if not isinstance(standardiser, Standardiser):
            raise ScopeError(f'expected a Standardiser, got {type(standardiser).__name__}')
        if standardiser.branch != branch:
            raise ScopeError(f'standardiser is for branch {standardiser.branch!r}, '
                             f'the data is {branch!r}')
        if standardiser.scope_kind != scope_kind:
            raise ScopeError(f'standardiser scope is {standardiser.scope_kind!r} '
                             f'({standardiser.scope!r}), this call needs '
                             f'{scope_kind!r}')
        if scope_kind == 'fold' and standardiser.fold != fold:
            raise ScopeError(f'standardiser was fitted for fold {standardiser.fold}, '
                             f'the data is fold {fold}')
        return standardiser

    def _restrict(self, requested: Optional[Sequence[str]], allowed: Sequence[str],
                  label: str) -> List[str]:
        if requested is None:
            return list(allowed)
        requested = [str(c) for c in requested]
        permitted = set(allowed)
        stray = sorted(set(requested) - permitted)
        if stray:
            raise ScopeError(f'{len(stray)} clip ids do not belong to {label}: '
                             f'{stray[:5]}')
        duplicates = len(requested) != len(set(requested))
        if duplicates:
            raise ScopeError(f'duplicate clip ids passed for {label}')
        return sorted(requested)

    def make_fold_batch(self, fold: int, split: str, branch: str,
                        clip_ids: Sequence[str]) -> Dict[str, Any]:
        '''ONE batch of clips from a specific fold and split, scaled by THAT
        fold's training-only standardiser.

        clip_ids is required. A fold training split holds 4,400 to 4,600 clips;
        at 1,759 features and up to 256 frames that is several gigabytes in one
        tensor, so there is deliberately no way to ask for a whole split as a
        single batch. Use iterate_fold_batches(..., batch_size=...) to walk a
        split in bounded mini-batches.

        There is no standardiser argument. The correct one is fetched here and
        checked, so no caller can substitute another fold's, the other branch's,
        the control's, or an ad-hoc probe's scaling.
        '''
        if split not in SPLITS:
            raise KeyError(f'{split!r} is not one of {SPLITS}')
        if clip_ids is None:
            raise ValueError(
                'make_fold_batch requires explicit clip_ids. Loading a whole split '
                'as one batch would allocate several gigabytes; use '
                'iterate_fold_batches(fold, split, branch, batch_size=...) instead.')
        ids = self._restrict(list(clip_ids), self.clip_ids(fold, split),
                             f'fold {fold} {split}')
        standardiser = self.require_standardiser_scope(
            self.fold_standardiser(fold, branch), branch, 'fold', fold)
        return self._make_batch(ids, branch, standardiser,
                                scope=f'fold:{fold}:{split}:{branch}')

    def make_control_batch(self, split: str, branch: str,
                           clip_ids: Sequence[str]) -> Dict[str, Any]:
        '''ONE batch of the signer-overlap control, scaled only by the control
        standardiser fitted on the 4,500 control-training clips.

        clip_ids is required, for the same memory reason as make_fold_batch. Use
        iterate_control_batches(..., batch_size=...) to walk a split. There is no
        standardiser argument.
        '''
        if split not in SPLITS:
            raise KeyError(f'{split!r} is not one of {SPLITS}')
        if clip_ids is None:
            raise ValueError(
                'make_control_batch requires explicit clip_ids. Loading a whole '
                'split as one batch would allocate several gigabytes; use '
                'iterate_control_batches(split, branch, batch_size=...) instead.')
        ids = self._restrict(list(clip_ids), self.control_clip_ids(split),
                             f'control {split}')
        standardiser = self.require_standardiser_scope(
            self.control_standardiser(branch), branch, 'control')
        return self._make_batch(ids, branch, standardiser,
                                scope=f'control:{split}:{branch}')

    def iterate_fold_batches(self, fold: int, split: str, branch: str,
                             batch_size: int, shuffle: bool = False,
                             seed: int = 42, epoch: int = 0,
                             clip_ids: Optional[Sequence[str]] = None):
        '''Walk a fold split in bounded mini-batches. This is the entry point
        training and evaluation loops should use: clip_ids=None means the whole
        split, but only ever batch_size clips are in memory at once.'''
        if int(batch_size) < 1:
            raise ValueError(f'batch_size must be at least 1, got {batch_size}')
        ids = self._restrict(clip_ids, self.clip_ids(fold, split),
                             f'fold {fold} {split}')
        order = epoch_order(ids, seed, epoch) if shuffle else sorted(ids)
        for start in range(0, len(order), batch_size):
            yield self.make_fold_batch(fold, split, branch,
                                       clip_ids=order[start:start + batch_size])

    def iterate_control_batches(self, split: str, branch: str, batch_size: int,
                                shuffle: bool = False, seed: int = 42,
                                epoch: int = 0,
                                clip_ids: Optional[Sequence[str]] = None):
        '''Walk a control split in bounded mini-batches. clip_ids=None means the
        whole split; only batch_size clips are in memory at once.'''
        if int(batch_size) < 1:
            raise ValueError(f'batch_size must be at least 1, got {batch_size}')
        ids = self._restrict(clip_ids, self.control_clip_ids(split),
                             f'control {split}')
        order = epoch_order(ids, seed, epoch) if shuffle else sorted(ids)
        for start in range(0, len(order), batch_size):
            yield self.make_control_batch(split, branch,
                                          clip_ids=order[start:start + batch_size])

    # ---- scope-safe RF access: original frames, no scaling, no padding -----
    def fold_raw_sequences(self, fold: int, split: str, branch: str,
                           clip_ids: Optional[Sequence[str]] = None):
        ids = self._restrict(clip_ids, self.clip_ids(fold, split),
                             f'fold {fold} {split}')
        for clip_id in ids:
            yield {'clip_id': clip_id, 'sequence': self.load_raw_sequence(clip_id, branch),
                   'label_id': self.label_id_for(clip_id),
                   'signer_id': self.signer_for(clip_id),
                   'frames': self.frames_for(clip_id)}

    def control_raw_sequences(self, split: str, branch: str,
                              clip_ids: Optional[Sequence[str]] = None):
        ids = self._restrict(clip_ids, self.control_clip_ids(split), f'control {split}')
        for clip_id in ids:
            yield {'clip_id': clip_id, 'sequence': self.load_raw_sequence(clip_id, branch),
                   'label_id': self.label_id_for(clip_id),
                   'signer_id': self.signer_for(clip_id),
                   'frames': self.frames_for(clip_id)}

    # ---- streaming sufficient statistics (the control path) -------------
    def stream_sufficient_statistics(self, clip_ids: Sequence[str], branch: str,
                                     workers: int = 4,
                                     progress: Optional[Callable[[int, int], None]] = None
                                     ) -> Tuple[int, np.ndarray, np.ndarray, int]:
        '''One pass, float64, nothing retained. Returns
        (frames, sum, sumsq, clips).

        Accumulation follows the sorted clip order regardless of worker count,
        because ThreadPoolExecutor.map yields results in input order. Float
        addition is order dependent, so this is what makes the control
        standardiser reproducible.
        '''
        _require_branch(branch)
        ordered = sorted(clip_ids)
        if len(set(ordered)) != len(ordered):
            raise ValueError('duplicate clip ids passed to the streaming pass')
        width = BRANCH_WIDTH[branch]
        frames = 0
        total_sum = np.zeros(width, dtype=np.float64)
        total_sumsq = np.zeros(width, dtype=np.float64)

        def one(clip_id: str):
            array = self.load_raw_sequence(clip_id, branch).astype(np.float64, copy=False)
            return int(array.shape[0]), array.sum(axis=0), np.square(array).sum(axis=0)

        done = 0
        with ThreadPoolExecutor(max_workers=max(1, int(workers))) as pool:
            for clip_frames, clip_sum, clip_sumsq in pool.map(one, ordered):
                frames += clip_frames
                total_sum += clip_sum
                total_sumsq += clip_sumsq
                done += 1
                if progress is not None and (done % 250 == 0 or done == len(ordered)):
                    progress(done, len(ordered))
        return frames, total_sum, total_sumsq, len(ordered)


def _require_branch(branch: str) -> None:
    if branch not in BRANCHES:
        raise KeyError(f'{branch!r} is not one of {BRANCHES}')


# ==========================================================================
# 9. Standardiser persistence
# ==========================================================================
def save_standardisers(path: Path, standardisers: Dict[str, Standardiser],
                       extra: Optional[Dict[str, Any]] = None) -> Path:
    branches = {s.branch for s in standardisers.values()}
    if len(branches) != 1:
        raise ValueError('one file holds one branch')
    branch = branches.pop()
    payload: Dict[str, np.ndarray] = {}
    metadata: Dict[str, Any] = {'loader_version': LOADER_VERSION,
                                'experiment_id': REQUIRED_EXPERIMENT_ID,
                                'branch': branch, 'entries': {}}
    for key, standardiser in standardisers.items():
        payload[f'{key}|mean'] = standardiser.mean
        payload[f'{key}|std'] = standardiser.std
        metadata['entries'][key] = standardiser.summary()
    payload['__keys__'] = np.array(sorted(standardisers))
    payload['__branch__'] = np.array([branch])
    payload['__experiment_id__'] = np.array([REQUIRED_EXPERIMENT_ID])
    payload['__loader_version__'] = np.array([LOADER_VERSION])
    payload['__metadata__'] = np.array([json.dumps(metadata | (extra or {}),
                                                   sort_keys=True, default=str)])
    path = Path(path)
    temporary = path.parent / (path.name + '.tmp')
    with open(temporary, 'wb') as handle:
        np.savez_compressed(handle, **payload)
    os.replace(temporary, path)
    return path


def load_standardisers(path: Path) -> Dict[str, Standardiser]:
    with np.load(Path(path), allow_pickle=False) as archive:
        branch = str(archive['__branch__'][0])
        experiment = str(archive['__experiment_id__'][0])
        if experiment != REQUIRED_EXPERIMENT_ID:
            raise ProvenanceError(
                f'{Path(path).name} was built for experiment {experiment}, '
                f'this loader requires {REQUIRED_EXPERIMENT_ID}')
        metadata = json.loads(str(archive['__metadata__'][0]))
        result: Dict[str, Standardiser] = {}
        for key in [str(k) for k in archive['__keys__']]:
            entry = metadata.get('entries', {}).get(key, {})
            recorded_fold = entry.get('fold')
            result[key] = Standardiser(
                branch=branch, scope=entry.get('scope', key),
                mean=np.asarray(archive[f'{key}|mean'], dtype=np.float64),
                std=np.asarray(archive[f'{key}|std'], dtype=np.float64),
                n_frames=int(entry.get('n_frames', 0)),
                fitted_on=entry.get('fitted_on', {}),
                scope_kind=str(entry.get('scope_kind', 'ad_hoc')),
                fold=None if recorded_fold is None else int(recorded_fold))
            recorded = entry.get('digest')
            if recorded and result[key].digest() != recorded:
                raise ProvenanceError(f'{key} in {Path(path).name} does not match '
                                      'its recorded digest')
    return result


# ==========================================================================
# 10. Module integrity
# ==========================================================================
def module_sha256(path: Optional[Any] = None) -> str:
    '''SHA-256 of the loader module file itself, full 64 hex characters.'''
    target = Path(path) if path is not None else Path(__file__)
    return hashlib.sha256(target.read_bytes()).hexdigest()


def verify_module(expected_sha256: str, path: Optional[Any] = None) -> str:
    '''Refuse to proceed if the published loader module has been edited.

    Downstream notebooks should hash the file BEFORE importing it, using the
    value recorded in loader_manifest.json:

        digest = hashlib.sha256(open(module_path, 'rb').read()).hexdigest()
        assert digest == manifest['module']['sha256']

    This function is the convenience form for code that has already imported it.
    '''
    actual = module_sha256(path)
    if actual != str(expected_sha256):
        raise ProvenanceError(
            f'loader module hash {actual} does not match the recorded '
            f'{expected_sha256}. The published data-access layer was modified; '
            'no downstream result computed with it is comparable.')
    return actual


# ==========================================================================
# 11. Provenance gate and context construction
# ==========================================================================
def _verify_schema(path: Path, branch: str) -> Dict[str, Any]:
    schema = json.loads(Path(path).read_text(encoding='utf-8'))
    if str(schema.get('variant')) != branch:
        raise ProvenanceError(f'{path.name} declares variant '
                              f'{schema.get("variant")!r}, expected {branch!r}')
    if str(schema.get('pipeline_id')) != REQUIRED_PIPELINE_ID:
        raise ProvenanceError(f'{path.name} pipeline id {schema.get("pipeline_id")!r} '
                              f'!= {REQUIRED_PIPELINE_ID!r}')
    if int(schema.get('features_per_frame', -1)) != BRANCH_WIDTH[branch]:
        raise ProvenanceError(f'{path.name} width {schema.get("features_per_frame")} '
                              f'!= {BRANCH_WIDTH[branch]}')
    blocks = {b['block']: (int(b['start']), int(b['end_exclusive']))
              for b in schema.get('blocks', [])}
    if blocks.get('face', (None, None))[1] != FACE_END[branch]:
        raise ProvenanceError(f'{path.name} face block ends at '
                              f'{blocks.get("face")}, expected {FACE_END[branch]}')
    if blocks.get('presence_mask') != MASK_SLICE[branch]:
        raise ProvenanceError(f'{path.name} presence_mask block is '
                              f'{blocks.get("presence_mask")}, expected '
                              f'{MASK_SLICE[branch]}')
    tail = BRANCH_WIDTH[branch] - FACE_END[branch]
    if tail != SHARED_TAIL_WIDTH:
        raise ProvenanceError(f'{branch} shared tail width {tail} != {SHARED_TAIL_WIDTH}')
    return schema


def build_context(search_root: str = '/kaggle/input',
                  verbose: bool = True) -> LoaderContext:
    '''Rediscover every attached root, verify the whole provenance chain, and
    return the trusted access layer. Raises rather than warns.'''
    say = (lambda *a: print(*a)) if verbose else (lambda *a: None)

    experiment_path = find_one('**/experiment_manifest.json',
                               'notebook 7 experiment manifest', search_root)
    nb7 = experiment_path.parent
    experiment = json.loads(experiment_path.read_text(encoding='utf-8'))

    if str(experiment.get('experiment_id')) != REQUIRED_EXPERIMENT_ID:
        raise ProvenanceError(f'attached experiment is '
                              f'{experiment.get("experiment_id")!r}, this loader '
                              f'requires {REQUIRED_EXPERIMENT_ID!r}')
    if str(experiment.get('manifest_version')) != REQUIRED_MANIFEST_VERSION:
        raise ProvenanceError(f'manifest version '
                              f'{experiment.get("manifest_version")!r} != '
                              f'{REQUIRED_MANIFEST_VERSION!r}')
    recorded = experiment.get('hashes', {})
    provenance_block = experiment.get('provenance', {})
    if str(provenance_block.get('pipeline_id')) != REQUIRED_PIPELINE_ID:
        raise ProvenanceError('experiment manifest pipeline id mismatch')
    if str(provenance_block.get('clip_table_hash')) != REQUIRED_CLIP_TABLE_HASH:
        raise ProvenanceError('experiment manifest clip table hash mismatch')
    if str(provenance_block.get('audit_verdict')) != REQUIRED_AUDIT_VERDICT:
        raise ProvenanceError('notebook 6 audit did not PASS')

    # ---- frozen tables, each required to reproduce its own hash ----------
    manifest, manifest_strategy, _ = read_frozen_csv(
        nb7 / 'master_manifest.csv', MANIFEST_HASH_COLUMNS,
        REQUIRED_HASHES['manifest'], MANIFEST_TEXT_COLUMNS)
    folds_table, fold_strategy, _ = read_frozen_csv(
        nb7 / 'fold_assignments.csv', FOLD_HASH_COLUMNS,
        REQUIRED_HASHES['fold_assignments'], FOLD_TEXT_COLUMNS)
    control_table, control_strategy, _ = read_frozen_csv(
        nb7 / 'CONTROL_signer_overlap_split.csv', CONTROL_HASH_COLUMNS,
        REQUIRED_HASHES['control_split'], CONTROL_TEXT_COLUMNS)

    label_map_raw = json.loads((nb7 / 'label_map.json').read_text(encoding='utf-8'))
    label_to_id = {str(k): int(v) for k, v in label_map_raw['label_to_id'].items()}
    label_hash = sha16(json.dumps(label_to_id, sort_keys=True, separators=(',', ':')))
    if label_hash != REQUIRED_HASHES['label_map']:
        raise ProvenanceError(f'label_map.json hashes to {label_hash}, '
                              f'required {REQUIRED_HASHES["label_map"]}')

    pairs_raw = json.loads((nb7 / 'signer_pairs.json').read_text(encoding='utf-8'))
    signer_pairs = [list(entry['signers']) for entry in pairs_raw['pairs']]
    pairs_hash = json_hash(signer_pairs)
    if pairs_hash != REQUIRED_HASHES['signer_pairs']:
        raise ProvenanceError(f'signer_pairs.json hashes to {pairs_hash}, '
                              f'required {REQUIRED_HASHES["signer_pairs"]}')

    pair_order = list(experiment['pairing']['final_cyclic_order'])
    order_hash = json_hash(pair_order)
    if order_hash != REQUIRED_HASHES['pair_order']:
        raise ProvenanceError(f'pair order hashes to {order_hash}, '
                              f'required {REQUIRED_HASHES["pair_order"]}')

    for name, expected in REQUIRED_HASHES.items():
        if str(recorded.get(name)) != expected:
            raise ProvenanceError(f'experiment_manifest.json records {name}='
                                  f'{recorded.get(name)!r}, required {expected!r}')

    # ---- recompute the experiment id from the reloaded components --------
    rebuilt = compute_experiment_id({
        'clip_table': REQUIRED_CLIP_TABLE_HASH,
        'manifest': canonical_hash(manifest, MANIFEST_HASH_COLUMNS),
        'fold_assignments': canonical_hash(folds_table, FOLD_HASH_COLUMNS),
        'signer_pairs': pairs_hash, 'pair_order': order_hash,
        'control_split': canonical_hash(control_table, CONTROL_HASH_COLUMNS),
        'label_map': label_hash,
        'manifest_version': str(experiment.get('manifest_version')),
    })
    if rebuilt != REQUIRED_EXPERIMENT_ID:
        raise ProvenanceError(f'the attached artefacts rebuild experiment id '
                              f'{rebuilt}, not {REQUIRED_EXPERIMENT_ID}')

    # ---- notebook 6 audit summary and statistics -------------------------
    audit_path = find_one('**/post_normalization_audit.json',
                          'notebook 6 audit summary', search_root)
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    audit_expectations = {
        'verdict': (str(audit.get('verdict')), REQUIRED_AUDIT_VERDICT),
        'mode': (str(audit.get('mode')), 'full'),
        'pipeline_id': (str(audit.get('pipeline_id')), REQUIRED_PIPELINE_ID),
        'clip_table_hash': (str(audit.get('clip_table_hash')), REQUIRED_CLIP_TABLE_HASH),
        'successful_clip_id_hash': (str(audit.get('successful_clip_id_hash')),
                                    REQUIRED_SUCCESSFUL_CLIP_ID_HASH),
    }
    wrong = {k: v for k, v in audit_expectations.items() if v[0] != v[1]}
    if wrong:
        detail = '; '.join(f'{k}: found {f!r}, required {e!r}'
                           for k, (f, e) in wrong.items())
        raise ProvenanceError(f'{audit_path.name} is not the verified audit. {detail}')

    statistics_paths, schemas, branch_roots = {}, {}, {}
    for branch in BRANCHES:
        statistics_paths[branch] = find_one(
            f'**/signer_feature_stats_{branch}.npz',
            f'notebook 6 {branch} statistics', search_root)
        schemas[branch] = _verify_schema(
            find_one(f'**/normalized_feature_schema_{branch}.json',
                     f'phase 5 {branch} schema', search_root), branch)
        branch_roots[branch] = find_one(f'**/normalized_landmarks/{branch}',
                                        f'phase 5 {branch} landmark root', search_root)

    # ---- shape the manifest ---------------------------------------------
    manifest = manifest.copy()
    manifest['clip_id'] = manifest['clip_id'].astype(str)
    manifest['relative_path'] = manifest['relative_path'].map(normalise_relative)
    manifest = manifest.sort_values('clip_id', kind='mergesort').reset_index(drop=True)
    if len(manifest) != N_CLIPS or manifest['clip_id'].nunique() != N_CLIPS:
        raise ProvenanceError(f'{len(manifest):,} manifest rows, required {N_CLIPS:,}')
    if manifest['label_id'].nunique() != N_LABELS:
        raise ProvenanceError('manifest does not hold 200 labels')
    if manifest['signer_id'].nunique() != N_SIGNERS:
        raise ProvenanceError('manifest does not hold 16 signers')
    if set(zip(manifest['label'], manifest['label_id'])) != set(label_to_id.items()):
        raise ProvenanceError('manifest label ids disagree with label_map.json')

    folds_table = folds_table.copy()
    folds_table['clip_id'] = folds_table['clip_id'].astype(str)
    folds_table['signer_id'] = folds_table['signer_id'].astype(str)
    folds_table['split'] = folds_table['split'].astype(str)
    control_table = control_table.copy()
    control_table['clip_id'] = control_table['clip_id'].astype(str)
    control_table['split'] = control_table['split'].astype(str)

    # ---- rebuild the folds from fold_assignments.csv alone ---------------
    folds = reconstruct_folds(folds_table)
    declared = [list(p) for p in experiment['pairing']['final_cyclic_pairs']]
    for fold in range(N_FOLDS):
        test_expected = tuple(sorted(declared[fold], key=_signer_number))
        validation_expected = tuple(sorted(declared[(fold + 1) % N_FOLDS],
                                           key=_signer_number))
        if folds[fold]['test'] != test_expected:
            raise ProvenanceError(f'fold {fold} test signers rebuilt as '
                                  f'{folds[fold]["test"]}, manifest declares '
                                  f'{test_expected}')
        if folds[fold]['validation'] != validation_expected:
            raise ProvenanceError(f'fold {fold} validation signers rebuilt as '
                                  f'{folds[fold]["validation"]}, manifest declares '
                                  f'{validation_expected}')

    provenance = {
        'loader_version': LOADER_VERSION,
        'experiment_id': REQUIRED_EXPERIMENT_ID,
        'manifest_version': REQUIRED_MANIFEST_VERSION,
        'pipeline_id': REQUIRED_PIPELINE_ID,
        'clip_table_hash': REQUIRED_CLIP_TABLE_HASH,
        'successful_clip_id_hash': REQUIRED_SUCCESSFUL_CLIP_ID_HASH,
        'experiment_id_recomputed': rebuilt,
        'hashes_verified': dict(REQUIRED_HASHES),
        'csv_read_strategy': {'master_manifest': manifest_strategy,
                              'fold_assignments': fold_strategy,
                              'control_split': control_strategy},
        'resolved': {
            'notebook7_dir': str(nb7), 'audit_summary': str(audit_path),
            'statistics': {b: str(p) for b, p in statistics_paths.items()},
            'branch_roots': {b: str(p) for b, p in branch_roots.items()},
        },
    }

    say(f'  experiment id      : {REQUIRED_EXPERIMENT_ID}  (rebuilt {rebuilt})')
    say(f'  pipeline id        : {REQUIRED_PIPELINE_ID}')
    say(f'  clip table hash    : {REQUIRED_CLIP_TABLE_HASH}')
    say(f'  successful clips   : {REQUIRED_SUCCESSFUL_CLIP_ID_HASH}')
    say(f'  csv read strategy  : {provenance["csv_read_strategy"]}')

    return LoaderContext(
        search_root=search_root, branch_roots=branch_roots, schemas=schemas,
        statistics_paths=statistics_paths, manifest=manifest,
        fold_assignments=folds_table, control_assignments=control_table,
        label_to_id=label_to_id,
        id_to_label={v: k for k, v in label_to_id.items()},
        folds=folds, signer_pairs=signer_pairs, pair_order=pair_order,
        provenance=provenance)


def reconstruct_folds(fold_assignments: pd.DataFrame) -> Dict[int, Dict[str, Tuple[str, ...]]]:
    '''Derive every fold's signer membership from fold_assignments.csv alone,
    then check the structural guarantees. Nothing is taken from the recorded
    pairing - that comparison happens in build_context, so two independent
    sources have to agree.'''
    folds: Dict[int, Dict[str, Tuple[str, ...]]] = {}
    present = sorted(int(f) for f in fold_assignments['fold'].unique())
    if present != list(range(N_FOLDS)):
        raise ProvenanceError(f'folds present are {present}, required 0..{N_FOLDS - 1}')

    test_counts: Dict[str, int] = {}
    validation_counts: Dict[str, int] = {}
    for fold in present:
        subset = fold_assignments[fold_assignments['fold'] == fold]
        if subset['clip_id'].nunique() != N_CLIPS or len(subset) != N_CLIPS:
            raise ProvenanceError(f'fold {fold} does not list every clip exactly once')
        membership = {}
        for split in SPLITS:
            signers = tuple(sorted(
                subset.loc[subset['split'] == split, 'signer_id'].astype(str).unique(),
                key=_signer_number))
            membership[split] = signers
        sizes = {'train': TRAIN_SIGNERS_PER_FOLD,
                 'validation': VALIDATION_SIGNERS_PER_FOLD,
                 'test': TEST_SIGNERS_PER_FOLD}
        for split, size in sizes.items():
            if len(membership[split]) != size:
                raise ProvenanceError(f'fold {fold} {split} has '
                                      f'{len(membership[split])} signers, required {size}')
        pooled = set(membership['train']) | set(membership['validation']) | set(membership['test'])
        if len(pooled) != N_SIGNERS:
            raise ProvenanceError(f'fold {fold} covers {len(pooled)} signers, required {N_SIGNERS}')
        for a in SPLITS:
            for b in SPLITS:
                if a < b and set(membership[a]) & set(membership[b]):
                    raise ProvenanceError(f'fold {fold}: {a} and {b} share signers')
        for split in SPLITS:
            classes = subset.loc[subset['split'] == split, 'label_id'].nunique()
            if classes != N_LABELS:
                raise ProvenanceError(f'fold {fold} {split} covers {classes} classes, '
                                      f'required {N_LABELS}')
        for signer in membership['test']:
            test_counts[signer] = test_counts.get(signer, 0) + 1
        for signer in membership['validation']:
            validation_counts[signer] = validation_counts.get(signer, 0) + 1
        folds[fold] = membership

    bad_test = {s: c for s, c in test_counts.items() if c != 1}
    bad_validation = {s: c for s, c in validation_counts.items() if c != 1}
    if bad_test or len(test_counts) != N_SIGNERS:
        raise ProvenanceError(f'signers not tested exactly once: {bad_test or test_counts}')
    if bad_validation or len(validation_counts) != N_SIGNERS:
        raise ProvenanceError(f'signers not validated exactly once: '
                              f'{bad_validation or validation_counts}')
    return folds


def loader_contract(extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    '''The data and loader contract. Deliberately free of model hyperparameters.'''
    contract = {
        'loader_version': LOADER_VERSION,
        'experiment_id': REQUIRED_EXPERIMENT_ID,
        'pipeline_id': REQUIRED_PIPELINE_ID,
        'clip_table_hash': REQUIRED_CLIP_TABLE_HASH,
        'successful_clip_id_hash': REQUIRED_SUCCESSFUL_CLIP_ID_HASH,
        'folds': {'n_folds': N_FOLDS, 'per_fold': '12 train / 2 validation / 2 test '
                                                  'signers, read from '
                                                  'fold_assignments.csv, never regenerated'},
        'branch_widths': dict(BRANCH_WIDTH),
        'face_block_end': dict(FACE_END),
        'presence_mask_columns': {b: list(MASK_SLICE[b]) for b in BRANCHES},
        'shared_tail_width': SHARED_TAIL_WIDTH,
        'path_reconstruction': 'relative_path is authoritative; absolute paths in '
                               'master_manifest.csv are stale and are never read',
        'random_forest_path': 'original variable-length sequences, no 256-frame '
                              'reduction, no padding, no frame-level z-scoring '
                              'required; derived-feature scaling, selection and PCA '
                              'are out of scope here and must be fitted on the '
                              'training signers inside the model notebook',
        'neural_length_policy': f'T <= {MAX_FRAMES}: every original frame is kept. '
                                f'T > {MAX_FRAMES}: {MAX_FRAMES} indices from '
                                'rint(linspace(0, T-1, 256)), strictly increasing, '
                                'first and last frame retained. No stretching, '
                                'repetition, interpolation, tail truncation or '
                                'first-N window.',
        'padding': f'dynamic per batch to the batch maximum, pad value {PAD_VALUE}, '
                   'applied after standardisation - never padded to MAX_FRAMES',
        'batch_returns': ['sequence float32 (B, Lmax, D)', 'lengths int64 (B,)',
                          'temporal_mask bool (B, Lmax), True = real frame',
                          'label_id int64 (B,)', 'clip_id', 'signer_id'],
        'temporal_mask_vs_presence_mask': 'the temporal mask says whether a frame '
                                          'exists; the four presence-mask columns say '
                                          'whether face / pose / left hand / right hand '
                                          'were detected in a real frame. They are '
                                          'different objects and both are provided.',
        'recurrent_requirement': 'neural notebooks MUST consume sequence + lengths via '
                                 'pack_padded_sequence(..., enforce_sorted=False) or a '
                                 'demonstrably equivalent masking implementation, so '
                                 'bidirectional recurrence can never process padded '
                                 'steps. Pooling over time must divide by true lengths.',
        'presence_mask_policy': 'presence-mask columns are never z-scored; mean is '
                                'forced to 0 and std to 1 so they stay strictly binary',
        'standardisation': {
            'formula': ['mean = sum / frames',
                        'variance = sumsq / frames - mean^2',
                        'std = sqrt(max(variance, 0))',
                        f'std[std < {STD_FLOOR}] = 1',
                        'mean[mask] = 0, std[mask] = 1'],
            'primary_folds': 'the 12 training signers of the fold only, summed from '
                             'the notebook 6 per-signer sufficient statistics; '
                             'validation and test signers are refused by an assertion',
            'control': f'streamed over the {CONTROL_TRAIN_CLIPS:,} control-training '
                       'clips only, because per-signer aggregates would carry the '
                       'control test clips of the same signers',
            'accumulation': 'float64, fixed order (signers by number, clips sorted), '
                            'so the result is bit-reproducible',
            'scope': 'frame-level features only',
        },
        'determinism': 'no random state in loading or reduction; shuffling only via '
                       'epoch_order(clip_ids, seed, epoch); repeated calls return '
                       'bit-identical arrays',
        'procedural_fairness': 'within a given model, the full and reduced branches '
                               'must be trained by an identical procedure. Optimiser, '
                               'learning rate, epochs, patience, dropout, architecture '
                               'and Random Forest settings are NOT part of this '
                               'contract: they are frozen in the respective model '
                               'notebook before any test evaluation. BiLSTM and BiGRU '
                               'need not match each other.',
        'safe_api': {
            'primary': 'iterate_fold_batches(fold, split, branch, batch_size=...) is '
                       'the entry point for training and evaluation; '
                       'make_fold_batch(fold, split, branch, clip_ids) builds a single '
                       'batch. Both fetch that fold\'s training-only standardiser '
                       'internally and expose no way to substitute another one.',
            'control': 'iterate_control_batches(split, branch, batch_size=...) and '
                       'make_control_batch(split, branch, clip_ids) use only the '
                       'control standardiser fitted on the control-training clips.',
            'random_forest': 'fold_raw_sequences(fold, split, branch) and '
                             'control_raw_sequences(split, branch) yield original '
                             'unscaled frames one clip at a time.',
            'subsetting': 'clip_ids may narrow a split but never widen it; any id '
                          'outside the requested fold and split raises ScopeError.',
            'explicit_batches': 'make_fold_batch and make_control_batch require '
                                'explicit clip_ids and refuse None. A fold training '
                                'split is 4,400 to 4,600 clips, which at 1,759 '
                                'features and up to 256 frames is several gigabytes '
                                'in one tensor, so a whole split can only be reached '
                                'through the bounded iterators.',
            'no_override': 'the safe methods take no standardiser argument. The '
                           'unguarded helpers _make_batch and _iterate_batches are '
                           'private, accept any standardiser for any clips, and must '
                           'not be called by notebooks 9 to 13. '
                           'require_standardiser_scope stays available for self-tests.',
            'obligation': 'notebooks 10 to 12 use the scope-safe API for every batch '
                          'they train or evaluate on.',
        },
        'torch_bridge': {
            'device_exempt': list(DEVICE_EXEMPT_KEYS),
            'rule': 'to_torch moves sequence, temporal_mask and label_id to the '
                    'requested device and leaves lengths on the CPU as int64, '
                    'because pack_padded_sequence requires CPU lengths.',
        },
        'module_verification': 'the sha256 of bsl_sequence_loader.py is recorded in '
                               'loader_manifest.json under module.sha256 and must be '
                               'checked before the module is imported downstream',
        'contract_id_scope': 'the loader contract id covers policy, boundaries and '
                             'verified provenance only. Generation time, resolved '
                             'session paths and runtime metadata are recorded '
                             'separately under run_metadata and are excluded, so the '
                             'same loader rerun later reproduces the same id.',
        'forbidden': ['regenerating folds', 'fitting any transformation on validation '
                      'or test signers', 'reading full_path or reduced_path',
                      'padding to a fixed length', 'z-scoring the presence masks',
                      'pairing data from one scope with another scope\'s standardiser',
                      'moving lengths off the CPU',
                      'requesting a whole split as a single batch',
                      'calling _make_batch or _iterate_batches from a model notebook'],
    }
    if extra:
        contract.update(extra)
    return contract


def contract_id(contract: Dict[str, Any]) -> str:
    '''Stable 16-hex id of the scientific contract. Volatile content is rejected
    rather than silently hashed, so the id cannot drift between reruns.'''
    assert_stable_contract(contract)
    return sha16(json.dumps(contract, sort_keys=True, separators=(',', ':'),
                            default=str))


def _iter_keys(node: Any):
    if isinstance(node, dict):
        for key, value in node.items():
            yield str(key)
            yield from _iter_keys(value)
    elif isinstance(node, (list, tuple)):
        for item in node:
            yield from _iter_keys(item)


def assert_stable_contract(contract: Dict[str, Any]) -> Dict[str, Any]:
    '''Refuse a contract carrying anything that changes between runs.'''
    volatile = sorted(set(_iter_keys(contract)) & VOLATILE_CONTRACT_KEYS)
    if volatile:
        raise ValueError('the loader contract must not carry run-specific keys: '
                         f'{volatile}. Record them under run_metadata instead.')
    blob = json.dumps(contract, sort_keys=True, default=str)
    if '/kaggle' in blob:
        raise ValueError('the loader contract must not carry session absolute paths')
    found = _TIMESTAMP_PATTERN.search(blob)
    if found:
        raise ValueError(f'the loader contract must not carry a timestamp '
                         f'({found.group(0)})')
    return contract


def standardiser_set_id(digests: Dict[str, Any]) -> str:
    '''Separate id over the actual fitted scalings, so downstream notebooks can
    verify the numbers as well as the policy.'''
    return sha16(json.dumps(digests, sort_keys=True, separators=(',', ':')))


__all__ = [
    'LOADER_VERSION', 'REQUIRED_EXPERIMENT_ID', 'REQUIRED_PIPELINE_ID',
    'REQUIRED_CLIP_TABLE_HASH', 'REQUIRED_SUCCESSFUL_CLIP_ID_HASH',
    'REQUIRED_HASHES', 'BRANCHES', 'BRANCH_WIDTH', 'FACE_END', 'MASK_SLICE',
    'SHARED_TAIL_WIDTH', 'MAX_FRAMES', 'N_FOLDS', 'N_CLIPS', 'N_LABELS',
    'N_SIGNERS', 'CONTROL_TRAIN_CLIPS', 'ProvenanceError', 'LeakageError',
    'canonical_hash', 'compute_experiment_id', 'array_digest', 'sha16',
    'json_hash', 'find_all', 'find_one', 'normalise_relative', 'read_frozen_csv',
    'SignerStatistics', 'load_signer_statistics', 'Standardiser',
    'standardiser_from_totals', 'sum_signer_statistics', 'uniform_indices',
    'collate_variable_length', 'to_torch', 'epoch_order', 'LoaderContext',
    'build_context', 'reconstruct_folds', 'save_standardisers',
    'load_standardisers', 'loader_contract', 'contract_id',
    'ScopeError', 'DEVICE_EXEMPT_KEYS', 'VOLATILE_CONTRACT_KEYS',
    'assert_stable_contract', 'standardiser_set_id', 'module_sha256',
    'verify_module',
]
"""[1:]


MODULE_PATH.write_text(MODULE_SOURCE, encoding="utf-8")

# Refuse to publish a module that does not even compile.
compile(MODULE_PATH.read_text(encoding="utf-8"), str(MODULE_PATH), "exec")

if str(WORKING) not in sys.path:
    sys.path.insert(0, str(WORKING))
if "bsl_sequence_loader" in sys.modules:
    del sys.modules["bsl_sequence_loader"]
import bsl_sequence_loader as L
importlib.reload(L)

print(f"wrote {MODULE_PATH.name}  "
      f"{MODULE_PATH.stat().st_size:,} bytes, "
      f"{len(MODULE_SOURCE.splitlines()):,} lines")
print(f"imported from {L.__file__}")

# The module carries its own copy of the contract constants. If they ever drift
# from this notebook's, one of the two has been edited and neither can be trusted.
print("\nCONTRACT CONSTANTS: notebook vs imported module")
constant_checks = [
    ("loader version", L.LOADER_VERSION, LOADER_VERSION),
    ("experiment id", L.REQUIRED_EXPERIMENT_ID, REQUIRED_EXPERIMENT_ID),
    ("manifest version", L.REQUIRED_MANIFEST_VERSION, REQUIRED_MANIFEST_VERSION),
    ("pipeline id", L.REQUIRED_PIPELINE_ID, REQUIRED_PIPELINE_ID),
    ("clip table hash", L.REQUIRED_CLIP_TABLE_HASH, REQUIRED_CLIP_TABLE_HASH),
    ("successful clip hash", L.REQUIRED_SUCCESSFUL_CLIP_ID_HASH,
     REQUIRED_SUCCESSFUL_CLIP_ID_HASH),
    ("frozen hashes", L.REQUIRED_HASHES, REQUIRED_HASHES),
    ("branch widths", L.BRANCH_WIDTH, EXPECTED["widths"]),
    ("face block end", L.FACE_END, EXPECTED["face_end"]),
    ("presence mask columns", L.MASK_SLICE, EXPECTED["mask_slice"]),
    ("shared tail width", L.SHARED_TAIL_WIDTH, EXPECTED["shared_tail_width"]),
    ("max frames", L.MAX_FRAMES, EXPECTED["max_frames"]),
    ("clips", L.N_CLIPS, EXPECTED["clips"]),
    ("labels", L.N_LABELS, EXPECTED["labels"]),
    ("signers", L.N_SIGNERS, EXPECTED["signers"]),
    ("folds", L.N_FOLDS, EXPECTED["folds"]),
    ("training signers per fold", L.TRAIN_SIGNERS_PER_FOLD, EXPECTED["train_signers"]),
    ("control training clips", L.CONTROL_TRAIN_CLIPS, EXPECTED["control_train_clips"]),
]
for name, found, wanted in constant_checks:
    require(f"module constant: {name}", found == wanted, f"{found!r}")

# The contract block must not smuggle a model hyperparameter into notebook 8.
FORBIDDEN_CONTRACT_KEYS = {
    "learning_rate", "lr", "optimizer", "optimiser", "epochs", "max_epochs",
    "patience", "dropout", "hidden_size", "num_layers", "n_estimators",
    "max_depth", "batch_size", "weight_decay", "momentum", "scheduler",
}


def _refuses_volatile(module) -> bool:
    """The hasher must reject a contract polluted with runtime metadata."""
    try:
        module.contract_id(dict(module.loader_contract(),
                                generated_at_utc="2026-01-01T00:00:00+00:00"))
    except ValueError:
        return True
    return False


def _all_keys(node: Any):
    if isinstance(node, dict):
        for key, value in node.items():
            yield str(key)
            yield from _all_keys(value)
    elif isinstance(node, list):
        for item in node:
            yield from _all_keys(item)


CONTRACT_PREVIEW = L.loader_contract()
smuggled = sorted(set(_all_keys(CONTRACT_PREVIEW)) & FORBIDDEN_CONTRACT_KEYS)
require("loader contract holds no model hyperparameter", not smuggled,
        smuggled or "data and loader terms only")
require("loader contract states the packing requirement",
        "enforce_sorted=False" in CONTRACT_PREVIEW["recurrent_requirement"],
        "pack_padded_sequence(..., enforce_sorted=False)")
require("loader contract states branch-procedural fairness",
        "identical procedure" in CONTRACT_PREVIEW["procedural_fairness"],
        "full vs reduced within the same model")
require("loader contract states the scope-safe API obligation",
        "no way to substitute" in CONTRACT_PREVIEW["safe_api"]["primary"]
        and "scope-safe API" in CONTRACT_PREVIEW["safe_api"]["obligation"],
        "iterate_fold_batches / iterate_control_batches")
require("loader contract requires explicit clip ids for single batches",
        "refuse None" in CONTRACT_PREVIEW["safe_api"]["explicit_batches"],
        "no whole split as one batch")
require("loader contract declares the unguarded helpers private",
        "_make_batch" in CONTRACT_PREVIEW["safe_api"]["no_override"]
        and "take no standardiser argument" in CONTRACT_PREVIEW["safe_api"]["no_override"],
        "no standardiser override")
require("loader contract states the CPU-lengths rule",
        CONTRACT_PREVIEW["torch_bridge"]["device_exempt"] == ["lengths"]
        and "CPU lengths" in CONTRACT_PREVIEW["torch_bridge"]["rule"],
        "lengths stay on the CPU")
require("loader contract states the module-hash obligation",
        "sha256" in CONTRACT_PREVIEW["module_verification"], "module.sha256")

# The contract id must be reproducible: no timestamp, no session path, no runtime
# metadata inside the hashed block, so the same loader rerun tomorrow gives the
# same id. The module refuses to hash a contract that carries any of those.
L.assert_stable_contract(CONTRACT_PREVIEW)
require("loader contract carries no run-specific content",
        not (set(_all_keys(CONTRACT_PREVIEW)) & set(L.VOLATILE_CONTRACT_KEYS)),
        "no timestamps, paths or environment")
require("loader contract id is reproducible across calls",
        L.contract_id(L.loader_contract()) == L.contract_id(L.loader_contract()),
        L.contract_id(CONTRACT_PREVIEW))
require("a run-specific key is refused by the contract hasher",
        _refuses_volatile(L), "ValueError on generated_at_utc")
require("the device exemption is declared by the module",
        tuple(L.DEVICE_EXEMPT_KEYS) == ("lengths",), L.DEVICE_EXEMPT_KEYS)
require("the scope guard exception exists",
        issubclass(L.ScopeError, RuntimeError), "ScopeError")

In [ ]:
# ============================================================
# CELL 3 — Provenance gate: rediscover, re-hash, refuse mismatches
# ============================================================
# Nothing below trusts a path recorded by notebook 7. Every root is rediscovered
# in THIS session, every frozen table is required to reproduce its own hash, and
# the experiment id is recomputed from the reloaded components rather than read.

print("DISCOVERY AND VERIFICATION")
context = L.build_context(search_root=SEARCH_ROOT, verbose=True)
provenance = context.provenance

print("\nresolved this session")
print(f"  notebook 7 dir     : {provenance['resolved']['notebook7_dir']}")
print(f"  notebook 6 audit   : {provenance['resolved']['audit_summary']}")
for branch in L.BRANCHES:
    print(f"  {branch:<8} statistics : {provenance['resolved']['statistics'][branch]}")
    print(f"  {branch:<8} landmarks  : {provenance['resolved']['branch_roots'][branch]}")

require("experiment id matches the frozen experiment",
        provenance["experiment_id"] == REQUIRED_EXPERIMENT_ID,
        provenance["experiment_id"])
require("experiment id recomputed from the reloaded artefacts",
        provenance["experiment_id_recomputed"] == REQUIRED_EXPERIMENT_ID,
        provenance["experiment_id_recomputed"])
require("all six notebook 7 hashes reproduced",
        provenance["hashes_verified"] == REQUIRED_HASHES,
        ", ".join(sorted(REQUIRED_HASHES)))
require("pipeline id matches phase 5",
        provenance["pipeline_id"] == REQUIRED_PIPELINE_ID, provenance["pipeline_id"])
require("notebook 6 clip table hash matches",
        provenance["clip_table_hash"] == REQUIRED_CLIP_TABLE_HASH,
        provenance["clip_table_hash"])
require("notebook 6 successful clip hash matches",
        provenance["successful_clip_id_hash"] == REQUIRED_SUCCESSFUL_CLIP_ID_HASH,
        provenance["successful_clip_id_hash"])

# ---- schemas ---------------------------------------------------------------
print("\nPHASE 5 SCHEMAS (read from disk, not assumed)")
for branch in L.BRANCHES:
    schema = context.schemas[branch]
    blocks = {b["block"]: (int(b["start"]), int(b["end_exclusive"]))
              for b in schema["blocks"]}
    print(f"  {branch:<8} width={schema['features_per_frame']} "
          f"face_landmarks={schema['face_landmarks_kept']} "
          f"face={blocks['face']} mask={blocks['presence_mask']}")
    require(f"{branch} schema width", schema["features_per_frame"] == EXPECTED["widths"][branch],
            schema["features_per_frame"])
    require(f"{branch} face block ends at the frozen boundary",
            blocks["face"][1] == EXPECTED["face_end"][branch], blocks["face"])
    require(f"{branch} presence-mask block", blocks["presence_mask"] == EXPECTED["mask_slice"][branch],
            blocks["presence_mask"])
    require(f"{branch} shared non-face tail width",
            EXPECTED["widths"][branch] - EXPECTED["face_end"][branch]
            == EXPECTED["shared_tail_width"], EXPECTED["shared_tail_width"])
require("both branches expose the same non-face tail width",
        EXPECTED["widths"]["full"] - EXPECTED["face_end"]["full"]
        == EXPECTED["widths"]["reduced"] - EXPECTED["face_end"]["reduced"],
        f"{EXPECTED['shared_tail_width']} columns")

# ---- per-signer sufficient statistics -------------------------------------
print("\nNOTEBOOK 6 PER-SIGNER SUFFICIENT STATISTICS")
for branch in L.BRANCHES:
    statistics = context.signer_statistics(branch)
    print(f"  {branch:<8} signers={len(statistics.signers)} "
          f"width={statistics.width} tail_start={statistics.tail_start} "
          f"frames={statistics.total_frames:,} audit={statistics.provenance['audit_version']}")
    require(f"{branch} statistics cover 16 signers",
            len(statistics.signers) == EXPECTED["signers"], len(statistics.signers))
    require(f"{branch} statistics frame total matches the audit",
            statistics.total_frames == EXPECTED["total_frames"],
            f"{statistics.total_frames:,}")
    require(f"{branch} statistics verdict is PASS",
            statistics.provenance["audit_verdict"] == "PASS",
            statistics.provenance["audit_verdict"])
    require(f"{branch} statistics cover 6,000 successful clips",
            statistics.provenance["successful_clip_count"] == EXPECTED["clips"],
            f"{statistics.provenance['successful_clip_count']:,}")

full_stats = context.signer_statistics("full")
reduced_stats = context.signer_statistics("reduced")
require("per-signer frame counts agree across branches",
        all(full_stats.frames[s] == reduced_stats.frames[s] for s in full_stats.signers),
        "16 of 16")
tail_sum_diff = max(float(np.abs(full_stats.sums[s][EXPECTED["face_end"]["full"]:]
                                 - reduced_stats.sums[s][EXPECTED["face_end"]["reduced"]:]).max())
                    for s in full_stats.signers)
tail_sumsq_diff = max(float(np.abs(full_stats.sumsq[s][EXPECTED["face_end"]["full"]:]
                                   - reduced_stats.sumsq[s][EXPECTED["face_end"]["reduced"]:]).max())
                      for s in full_stats.signers)
require("shared-tail sums identical across branches", tail_sum_diff == 0.0,
        f"max diff {tail_sum_diff:.2e}")
require("shared-tail sums of squares identical across branches", tail_sumsq_diff == 0.0,
        f"max diff {tail_sumsq_diff:.2e}")

# ---- the manifest ----------------------------------------------------------
manifest = context.manifest
print(f"\nmanifest rows        : {len(manifest):,}")
print(f"unique clip ids      : {manifest['clip_id'].nunique():,}")
print(f"labels               : {manifest['label_id'].nunique()}")
print(f"signers              : {manifest['signer_id'].nunique()}")
print(f"frames               : {int(manifest['original_frames'].sum()):,} total | "
      f"{manifest['original_frames'].mean():.2f} mean | "
      f"{int(manifest['original_frames'].min())}-{int(manifest['original_frames'].max())} range")
long_clips = int((manifest["original_frames"] > EXPECTED["max_frames"]).sum())
print(f"clips over {EXPECTED['max_frames']} frames : {long_clips} "
      f"({100 * long_clips / len(manifest):.1f}%)")
print(f"csv read strategy    : {provenance['csv_read_strategy']}")

require("manifest holds 6,000 unique clips",
        len(manifest) == EXPECTED["clips"] and manifest["clip_id"].nunique() == EXPECTED["clips"],
        f"{len(manifest):,}")
require("manifest frame total matches the audit",
        int(manifest["original_frames"].sum()) == EXPECTED["total_frames"],
        f"{int(manifest['original_frames'].sum()):,}")
require("manifest holds 200 labels and 16 signers",
        manifest["label_id"].nunique() == EXPECTED["labels"]
        and manifest["signer_id"].nunique() == EXPECTED["signers"], "200 / 16")
require("every signer covers every class",
        int(manifest.groupby("signer_id")["label_id"].nunique().min()) == EXPECTED["labels"],
        "min 200")
require("label ids are a contiguous 0..199 range",
        sorted(manifest["label_id"].unique()) == list(range(EXPECTED["labels"])),
        "0..199")
require("stale absolute paths are ignored",
        "full_path" not in L.MANIFEST_HASH_COLUMNS
        and "reduced_path" not in L.MANIFEST_HASH_COLUMNS,
        "relative_path is authoritative")

In [ ]:
# ============================================================
# CELL 4 — Folds rebuilt from fold_assignments.csv, then cross-checked
# ============================================================
# build_context already derived the fold membership from fold_assignments.csv
# alone and compared it against the pairing recorded in experiment_manifest.json.
# Two independent sources had to agree. This cell reports the result and checks
# the properties the statistical plan depends on.

folds = context.folds
print("FOLD MEMBERSHIP (rebuilt from fold_assignments.csv, not from the recorded pairing)")
print(f"{'fold':>4}  {'test':<10} {'validation':<10} {'train':>5} {'val':>5} {'test':>5}")
fold_rows = []
for fold in range(EXPECTED["folds"]):
    definition = folds[fold]
    counts = {split: len(context.clip_ids(fold, split))
              for split in ("train", "validation", "test")}
    fold_rows.append({"fold": fold,
                      "test_signers": " ".join(definition["test"]),
                      "validation_signers": " ".join(definition["validation"]),
                      "train_signers": " ".join(definition["train"]),
                      **{f"{k}_clips": v for k, v in counts.items()}})
    print(f"{fold:>4}  {' '.join(definition['test']):<10} "
          f"{' '.join(definition['validation']):<10} "
          f"{counts['train']:>5} {counts['validation']:>5} {counts['test']:>5}")
fold_table = pd.DataFrame(fold_rows)

print(f"\ncyclic pair order    : {context.pair_order}")
print(f"signer pairs         : {['+'.join(p) for p in context.signer_pairs]}")

# ---- structural guarantees -------------------------------------------------
print("\nLEAKAGE AND STRUCTURE CHECKS")
test_counts: Dict[str, int] = {}
validation_counts: Dict[str, int] = {}
for fold in range(EXPECTED["folds"]):
    definition = folds[fold]
    for signer in definition["test"]:
        test_counts[signer] = test_counts.get(signer, 0) + 1
    for signer in definition["validation"]:
        validation_counts[signer] = validation_counts.get(signer, 0) + 1

require("every signer is a test signer exactly once",
        len(test_counts) == EXPECTED["signers"]
        and set(test_counts.values()) == {1}, f"{len(test_counts)} signers")
require("every signer is a validation signer exactly once",
        len(validation_counts) == EXPECTED["signers"]
        and set(validation_counts.values()) == {1}, f"{len(validation_counts)} signers")
require("every fold has 12 train / 2 validation / 2 test signers",
        all(len(folds[f]["train"]) == EXPECTED["train_signers"]
            and len(folds[f]["validation"]) == EXPECTED["validation_signers"]
            and len(folds[f]["test"]) == EXPECTED["test_signers"]
            for f in range(EXPECTED["folds"])), "8 of 8")

signer_disjoint = True
clip_disjoint = True
classes_complete = True
for fold in range(EXPECTED["folds"]):
    definition = folds[fold]
    sets = {split: set(definition[split]) for split in ("train", "validation", "test")}
    if (sets["train"] & sets["validation"]) or (sets["train"] & sets["test"]) \
            or (sets["validation"] & sets["test"]):
        signer_disjoint = False
    clip_sets = {split: set(context.clip_ids(fold, split))
                 for split in ("train", "validation", "test")}
    if (clip_sets["train"] & clip_sets["validation"]) or (clip_sets["train"] & clip_sets["test"]) \
            or (clip_sets["validation"] & clip_sets["test"]):
        clip_disjoint = False
    if sum(len(v) for v in clip_sets.values()) != EXPECTED["clips"]:
        clip_disjoint = False
    for split, clips in clip_sets.items():
        if context.manifest.loc[context.manifest["clip_id"].isin(clips),
                                "label_id"].nunique() != EXPECTED["labels"]:
            classes_complete = False

require("train / validation / test signers disjoint in all 8 folds", signer_disjoint, "8 of 8")
require("clip sets partition the dataset in every fold", clip_disjoint, "8 of 8")
require("all 200 classes present in every split of every fold", classes_complete, "24 of 24")

# ---- the pooled out-of-fold set the primary result rests on ----------------
pooled: List[str] = []
for fold in range(EXPECTED["folds"]):
    pooled.extend(context.clip_ids(fold, "test"))
require("pooled out-of-fold predictions cover every clip exactly once",
        len(pooled) == EXPECTED["clips"] and len(set(pooled)) == EXPECTED["clips"],
        f"{len(pooled):,} rows, {len(set(pooled)):,} unique")

# Each pooled prediction must come from a model that never trained on that signer.
oof_clean = True
for fold in range(EXPECTED["folds"]):
    train_signers = set(folds[fold]["train"])
    test_clips = context.clip_ids(fold, "test")
    test_signers = set(context.manifest.loc[
        context.manifest["clip_id"].isin(test_clips), "signer_id"])
    if train_signers & test_signers:
        oof_clean = False
require("no pooled test clip shares a signer with its own training set", oof_clean,
        "8 of 8 folds")

print(f"\ntrain clip counts    : {sorted(fold_table['train_clips'].tolist())}")
print(f"training-size spread : {int(fold_table['train_clips'].min())}"
      f"-{int(fold_table['train_clips'].max())} "
      f"(std {float(fold_table['train_clips'].std(ddof=0)):.1f})")
print("\nPairing reduced fold-to-fold imbalance attributable to the two predefined")
print("nuisance covariates (signer mean sequence length and hand-tracking quality).")
print("It does not remove other sources of between-fold variation.")

# ---- the signer-overlap control -------------------------------------------
control = context.control_assignments
control_counts = {split: len(context.control_clip_ids(split))
                  for split in ("train", "validation", "test")}
control_train_signers = set(control.loc[control["split"] == "train", "signer_id"])
control_test_signers = set(control.loc[control["split"] == "test", "signer_id"])
shared = control_train_signers & control_test_signers

print("\nSIGNER-OVERLAP CONTROL (one fixed split, invalid as a signer-independent test)")
print(f"  clips per split          : {control_counts}")
print(f"  signers shared train/test: {len(shared)} of {EXPECTED['signers']}")
print(f"  split type               : {sorted(control['split_type'].unique())}")
print("  The overlap is intentional. It measures the signer-overlap performance gap;")
print("  identity memorisation is one plausible contributor, session and background")
print("  similarity are others, so it is reported as a gap and not asserted to be leakage.")

require("control training split holds 4,500 clips",
        control_counts["train"] == EXPECTED["control_train_clips"], control_counts["train"])
require("control splits partition the dataset",
        sum(control_counts.values()) == EXPECTED["clips"]
        and control["clip_id"].nunique() == EXPECTED["clips"], sum(control_counts.values()))
require("control covers all 200 classes in every split",
        all(control.loc[control["split"] == s, "label_id"].nunique() == EXPECTED["labels"]
            for s in ("train", "validation", "test")), "3 of 3")
require("control shares all 16 signers between train and test",
        len(shared) == EXPECTED["signers"], f"{len(shared)} of {EXPECTED['signers']}")
require("control is labelled invalid in the file itself",
        set(control["split_type"].unique()) == {"signer_overlap_INVALID_control"},
        "signer_overlap_INVALID_control")

In [ ]:
# ============================================================
# CELL 5 — Path reconstruction and raw-loader validation
# ============================================================
# Absolute paths in master_manifest.csv belong to the notebook 7 runtime. Every
# path here is rebuilt as {rediscovered branch root} / relative_path.

print("PATH RECONSTRUCTION (relative_path is the authoritative field)")
started = time.time()
missing: Dict[str, List[str]] = {branch: [] for branch in L.BRANCHES}
for branch in L.BRANCHES:
    root = context.branch_roots[branch]
    for clip_id, relative in zip(context.manifest["clip_id"],
                                 context.manifest["relative_path"]):
        if not (root / relative).exists():
            missing[branch].append(clip_id)
            if len(missing[branch]) > 5:
                break
    print(f"  {branch:<8} root {root}")
    print(f"           missing {len(missing[branch])} of {len(context.manifest):,}")

for branch in L.BRANCHES:
    require(f"{branch}: every reconstructed path exists", not missing[branch],
            missing[branch][:3] or f"{len(context.manifest):,} of {len(context.manifest):,}")
print(f"  existence sweep: {time.time() - started:.1f}s")

# ---- a deterministic deep sample -------------------------------------------
# Cheap where it can be, thorough where it matters: both extremes of the length
# distribution, every clip that triggers the uniform-reduction path, two clips
# from every signer, plus a seeded random draw.
lengths = context.manifest.set_index("clip_id")["original_frames"]
sample: List[str] = [str(lengths.idxmin()), str(lengths.idxmax())]
sample += context.manifest.loc[context.manifest["original_frames"] > EXPECTED["max_frames"],
                               "clip_id"].astype(str).tolist()
for signer, group in context.manifest.groupby("signer_id"):
    sample += sorted(group["clip_id"].astype(str))[:2]
sample += (context.manifest["clip_id"].astype(str)
           .sample(n=min(40, len(context.manifest)), random_state=RANDOM_SEED).tolist())
sample = sorted(dict.fromkeys(sample))

print(f"\nDEEP SAMPLE: {len(sample)} clips x 2 branches")
print(f"  shortest {lengths.min()} frames ({lengths.idxmin()}) | "
      f"longest {lengths.max()} frames ({lengths.idxmax()})")
print(f"  clips above {EXPECTED['max_frames']} frames included: "
      f"{int((context.manifest['original_frames'] > EXPECTED['max_frames']).sum())}")

width_bad, frame_bad, finite_bad, mask_bad, tail_numeric_bad, tail_byte_bad = [], [], [], [], [], []
for clip_id in sample:
    arrays = {}
    for branch in L.BRANCHES:
        array = context.load_raw_sequence(clip_id, branch)
        arrays[branch] = array
        if int(array.shape[1]) != EXPECTED["widths"][branch]:
            width_bad.append(f"{clip_id}/{branch}")
        if int(array.shape[0]) != int(lengths[clip_id]):
            frame_bad.append(f"{clip_id}/{branch}")
        if not np.isfinite(array).all():
            finite_bad.append(f"{clip_id}/{branch}")
        start, end = EXPECTED["mask_slice"][branch]
        if not np.isin(array[:, start:end], (0.0, 1.0)).all():
            mask_bad.append(f"{clip_id}/{branch}")
    full_tail = arrays["full"][:, EXPECTED["face_end"]["full"]:]
    reduced_tail = arrays["reduced"][:, EXPECTED["face_end"]["reduced"]:]
    if not np.array_equal(full_tail, reduced_tail):
        tail_numeric_bad.append(clip_id)
    if full_tail.tobytes() != reduced_tail.tobytes():
        tail_byte_bad.append(clip_id)

require("sampled clips have the correct branch width", not width_bad, width_bad[:3] or "clean")
require("sampled frame counts match the manifest", not frame_bad, frame_bad[:3] or "clean")
require("sampled clips are finite", not finite_bad, finite_bad[:3] or "clean")
require("sampled presence masks are strictly binary", not mask_bad, mask_bad[:3] or "clean")
# This is the check that catches a wrong branch offset in the loader itself, which
# would silently destroy the controlled full-vs-reduced comparison.
require("non-face columns numerically identical across branches (loader layer)",
        not tail_numeric_bad, tail_numeric_bad[:3] or f"{len(sample)} clips, 355 columns")
require("non-face columns byte identical across branches (loader layer)",
        not tail_byte_bad, tail_byte_bad[:3] or f"{len(sample)} clips")

# ---- the loader must reject rather than repair -----------------------------
print("\nNEGATIVE TESTS (the loader must refuse, not silently cope)")
probe = sample[0]


def refuses(name: str, call, expected_error) -> None:
    try:
        call()
    except expected_error as error:
        record(name, True, f"{type(error).__name__}")
        return
    except Exception as error:                                # wrong exception type
        record(name, False, f"raised {type(error).__name__} instead")
        raise RuntimeError(f"{name}: wrong exception type") from error
    record(name, False, "no error raised")
    raise RuntimeError(f"{name}: the loader accepted invalid input")


refuses("unknown branch rejected",
        lambda: context.load_raw_sequence(probe, "holistic"), KeyError)
refuses("unknown clip id rejected",
        lambda: context.load_raw_sequence("p99_z_nonexistent", "full"), KeyError)
refuses("mismatched standardiser branch rejected",
        lambda: context.neural_sequence(
            probe, "full", L.standardiser_from_totals(
                "reduced", 10, np.zeros(631), np.ones(631), scope="probe")),
        ValueError)
refuses("wrong-width sequence rejected by the standardiser",
        lambda: L.standardiser_from_totals(
            "full", 10, np.zeros(1759), np.ones(1759), scope="probe"
        ).apply(np.zeros((4, 631), dtype=np.float32)), ValueError)
refuses("zero-frame standardiser rejected",
        lambda: L.standardiser_from_totals("full", 0, np.zeros(1759), np.zeros(1759),
                                           scope="probe"), ValueError)

In [ ]:
# ============================================================
# CELL 6 — Leakage-safe fold standardisers: 8 folds x 2 branches
# ============================================================
#   mean     = sum / frames
#   variance = sumsq / frames - mean^2
#   std      = sqrt(max(variance, 0));  std[std < 1e-8] = 1
#   mean[presence-mask columns] = 0, std[...] = 1
#
# Summed over the 12 TRAINING signers of the fold only. The module raises
# LeakageError if a validation or test signer is ever passed in.

print("FOLD STANDARDISERS (12 training signers per fold, no validation or test)")
fold_standardisers: Dict[str, Dict[str, Any]] = {branch: {} for branch in L.BRANCHES}
standardiser_rows = []

for branch in L.BRANCHES:
    for fold in range(EXPECTED["folds"]):
        standardiser = context.fold_standardiser(fold, branch)
        if standardiser.scope_kind != "fold" or standardiser.fold != fold:
            raise RuntimeError(f"fold {fold} {branch} standardiser is tagged "
                               f"{standardiser.scope_kind!r}/{standardiser.fold}")
        fold_standardisers[branch][f"fold_{fold}"] = standardiser
        summary = standardiser.summary()
        standardiser_rows.append({
            "branch": branch, "fold": fold, "frames": summary["n_frames"],
            "mean_min": summary["mean_min"], "mean_max": summary["mean_max"],
            "std_min": summary["std_min"], "std_max": summary["std_max"],
            "floored_columns": summary["floored_columns"], "digest": summary["digest"],
        })
standardiser_table = pd.DataFrame(standardiser_rows)
print(standardiser_table.round(4).to_string(index=False))

# ---- the standardiser must have seen training signers only -----------------
leak_free = True
frames_consistent = True
for branch in L.BRANCHES:
    statistics = context.signer_statistics(branch)
    for fold in range(EXPECTED["folds"]):
        standardiser = fold_standardisers[branch][f"fold_{fold}"]
        fitted = set(standardiser.fitted_on["training_signers"])
        held_out = set(folds[fold]["validation"]) | set(folds[fold]["test"])
        if fitted != set(folds[fold]["train"]) or (fitted & held_out):
            leak_free = False
        expected_frames = sum(statistics.frames[s] for s in folds[fold]["train"])
        if standardiser.n_frames != expected_frames:
            frames_consistent = False

require("every fold standardiser was fitted on exactly its 12 training signers",
        leak_free, "16 of 16 standardisers")
require("standardiser frame counts equal the summed training-signer frames",
        frames_consistent, "16 of 16")
require("no standardiser saw all 16 signers",
        all(fold_standardisers[b][f"fold_{f}"].n_frames < EXPECTED["total_frames"]
            for b in L.BRANCHES for f in range(EXPECTED["folds"])),
        f"all below {EXPECTED['total_frames']:,} frames")

# ---- presence masks stay binary -------------------------------------------
mask_clean = True
for branch in L.BRANCHES:
    start, end = EXPECTED["mask_slice"][branch]
    for fold in range(EXPECTED["folds"]):
        standardiser = fold_standardisers[branch][f"fold_{fold}"]
        if not (np.all(standardiser.mean[start:end] == 0.0)
                and np.all(standardiser.std[start:end] == 1.0)):
            mask_clean = False
require("presence-mask columns have mean 0 and std 1 in every standardiser",
        mask_clean, "16 of 16")

# ---- the free exact invariant ---------------------------------------------
# The non-face columns are byte-identical across branches and both standardisers
# sum the same 12 signers, so the tail of the full standardiser must equal the
# tail of the reduced one exactly. An off-by-one in either branch offset fails
# here immediately, and costs nothing to check.
print("\nCROSS-BRANCH TAIL INVARIANCE  full[1404:] vs reduced[276:]")
mean_diffs, std_diffs = [], []
for fold in range(EXPECTED["folds"]):
    full_std = fold_standardisers["full"][f"fold_{fold}"]
    reduced_std = fold_standardisers["reduced"][f"fold_{fold}"]
    mean_diff = float(np.abs(full_std.mean[EXPECTED["face_end"]["full"]:]
                             - reduced_std.mean[EXPECTED["face_end"]["reduced"]:]).max())
    std_diff = float(np.abs(full_std.std[EXPECTED["face_end"]["full"]:]
                            - reduced_std.std[EXPECTED["face_end"]["reduced"]:]).max())
    mean_diffs.append(mean_diff)
    std_diffs.append(std_diff)
    print(f"  fold {fold}: mean max diff {mean_diff:.2e} | std max diff {std_diff:.2e} | "
          f"frames {full_std.n_frames:,} vs {reduced_std.n_frames:,}")

require("fold standardiser tail means identical across branches",
        max(mean_diffs) == 0.0, f"max {max(mean_diffs):.2e} over 8 folds")
require("fold standardiser tail stds identical across branches",
        max(std_diffs) == 0.0, f"max {max(std_diffs):.2e} over 8 folds")
require("fold standardiser training frame counts identical across branches",
        all(fold_standardisers["full"][f"fold_{f}"].n_frames
            == fold_standardisers["reduced"][f"fold_{f}"].n_frames
            for f in range(EXPECTED["folds"])), "8 of 8")

# ---- the leakage guard must actually fire ---------------------------------
print("\nLEAKAGE GUARD (must refuse, not warn)")
for fold in (0, 3):
    intruder = folds[fold]["test"][0]
    tampered = list(folds[fold]["train"])[:-1] + [intruder]
    try:
        context.fold_standardiser(fold, "full", signers=tampered)
        require(f"fold {fold}: a test signer in the fit is refused", False, "accepted")
    except L.LeakageError as error:
        record(f"fold {fold}: a test signer in the fit is refused", True, intruder)
    intruder = folds[fold]["validation"][0]
    tampered = list(folds[fold]["train"])[:-1] + [intruder]
    try:
        context.fold_standardiser(fold, "reduced", signers=tampered)
        require(f"fold {fold}: a validation signer in the fit is refused", False, "accepted")
    except L.LeakageError:
        record(f"fold {fold}: a validation signer in the fit is refused", True, intruder)
incomplete = list(folds[0]["train"])[:-1]
try:
    context.fold_standardiser(0, "full", signers=incomplete)
    require("a fit missing one training signer is refused", False, "accepted")
except L.LeakageError:
    record("a fit missing one training signer is refused", True,
           f"{len(incomplete)} of {EXPECTED['train_signers']} rejected")
try:
    context.fold_standardiser(0, "reduced",
                              signers=list(folds[0]["train"]) + [folds[0]["test"][0]])
    require("a 13th signer in the fit is refused", False, "accepted")
except L.LeakageError:
    record("a 13th signer in the fit is refused", True, folds[0]["test"][0])

# ---- persist ---------------------------------------------------------------
for branch in L.BRANCHES:
    path = L.save_standardisers(
        FOLD_STD_PATH[branch], fold_standardisers[branch],
        extra={"scope": "primary_folds",
               "fitted_on": "the 12 training signers of each fold, from the "
                            "notebook 6 per-signer sufficient statistics"})
    print(f"\nwrote {path.name}  {path.stat().st_size:,} bytes  "
          f"({EXPECTED['folds']} standardisers)")
    reloaded = L.load_standardisers(path)
    require(f"{branch} fold standardisers round-trip from disk",
            all(reloaded[k].digest() == fold_standardisers[branch][k].digest()
                for k in fold_standardisers[branch]),
            f"{len(reloaded)} entries")
    require(f"{branch} fold standardisers keep their scope across the round trip",
            all(reloaded[k].scope_kind == "fold"
                and reloaded[k].fold == fold_standardisers[branch][k].fold
                for k in fold_standardisers[branch]),
            "scope_kind=fold with the fold number")

In [ ]:
# ============================================================
# CELL 7 — Validate the sufficient statistics against the real files
# ============================================================
# Notebook 6 proved the combined-sums identity on synthetic arrays. This confirms
# it on the actual .npy files this loader will read, for both branches.
#
# Two training signers are selected deterministically as the two with the fewest
# clips (ties broken by signer number), which keeps the read small and covers a
# 200-clip signer if one is present in the fold. No clip count is assumed: every
# clip belonging to the selected signers is streamed and the actual number is
# reported.

VALIDATION_FOLD = 0
clip_counts = context.manifest.groupby("signer_id")["clip_id"].size()
candidates = sorted(folds[VALIDATION_FOLD]["train"],
                    key=lambda s: (int(clip_counts[s]), int(str(s).lstrip("p"))))
selected = candidates[:2]

expected_clips = {s: int(clip_counts[s]) for s in selected}
expected_frames = {s: int(context.manifest.loc[context.manifest["signer_id"] == s,
                                               "original_frames"].sum())
                   for s in selected}
print(f"DIRECT RECOMPUTATION  fold {VALIDATION_FOLD} training signers {selected}")
for signer in selected:
    print(f"  {signer}: {expected_clips[signer]} clips, "
          f"{expected_frames[signer]:,} frames (from the manifest)")
print(f"  total to stream: {sum(expected_clips.values())} clips x "
      f"{len(L.BRANCHES)} branches")

worst = {"sum": 0.0, "sumsq": 0.0, "mean": 0.0, "std": 0.0}
count_ok, frame_ok = True, True
started = time.time()

for branch in L.BRANCHES:
    statistics = context.signer_statistics(branch)
    for signer in selected:
        clips = sorted(context.manifest.loc[context.manifest["signer_id"] == signer,
                                            "clip_id"].astype(str))
        frames, total_sum, total_sumsq, streamed = context.stream_sufficient_statistics(
            clips, branch, workers=STREAM_WORKERS)

        if streamed != expected_clips[signer]:
            count_ok = False
        if frames != expected_frames[signer] or frames != statistics.frames[signer]:
            frame_ok = False

        sum_diff = float(np.abs(total_sum - statistics.sums[signer]).max())
        sumsq_diff = float(np.abs(total_sumsq - statistics.sumsq[signer]).max())

        # What actually matters is the transformation, so compare mean and std too.
        streamed_std = L.standardiser_from_totals(
            branch, frames, total_sum, total_sumsq, scope=f"validate:{signer}")
        published_std = L.standardiser_from_totals(
            branch, statistics.frames[signer], statistics.sums[signer],
            statistics.sumsq[signer], scope=f"published:{signer}")
        mean_diff = float(np.abs(streamed_std.mean - published_std.mean).max())
        std_diff = float(np.abs(streamed_std.std - published_std.std).max())

        worst["sum"] = max(worst["sum"], sum_diff)
        worst["sumsq"] = max(worst["sumsq"], sumsq_diff)
        worst["mean"] = max(worst["mean"], mean_diff)
        worst["std"] = max(worst["std"], std_diff)

        print(f"  {branch:<8} {signer:<4} streamed {streamed:>3} clips, "
              f"{frames:>7,} frames | sum {sum_diff:.2e} sumsq {sumsq_diff:.2e} "
              f"mean {mean_diff:.2e} std {std_diff:.2e}")

print(f"  streaming pass: {time.time() - started:.1f}s")

require("streamed clip counts match the manifest", count_ok,
        f"{sum(expected_clips.values())} clips")
require("streamed frame counts match the manifest and the notebook 6 statistics",
        frame_ok, f"{sum(expected_frames.values()):,} frames")
# float64 summation order differs between notebook 6's worker pool and this pass,
# so the raw totals are compared with a tolerance and the derived transformation
# with a tight one. The transformation is what the models actually consume.
require("recomputed sums agree with the published statistics",
        worst["sum"] < 1e-4 and worst["sumsq"] < 1e-3,
        f"sum {worst['sum']:.2e}, sumsq {worst['sumsq']:.2e}")
require("recomputed mean and std agree with the published statistics",
        worst["mean"] < 1e-8 and worst["std"] < 1e-8,
        f"mean {worst['mean']:.2e}, std {worst['std']:.2e}")

In [ ]:
# ============================================================
# CELL 8 — Control standardisers, from the 4,500 control-training clips only
# ============================================================
# The signer-overlap control shares every signer between train and test, so the
# notebook 6 per-signer aggregates would carry the control's own test clips into
# the scaling. They cannot be used here. One streaming float64 pass over the
# control-training clips is the only leakage-safe option.
#
# Accumulation follows the sorted clip order regardless of worker count, because
# ThreadPoolExecutor.map yields in input order. Float addition is order dependent,
# so this is what makes the result reproducible.

control_train = context.control_clip_ids("train")
control_expected_frames = int(context.manifest.loc[
    context.manifest["clip_id"].isin(control_train), "original_frames"].sum())

print(f"CONTROL STANDARDISERS  ({len(control_train):,} training clips per branch)")
print(f"  expected frames : {control_expected_frames:,} "
      f"({100 * control_expected_frames / EXPECTED['total_frames']:.1f}% of the dataset)")
print(f"  workers         : {STREAM_WORKERS}")
require("control training set holds exactly 4,500 clips",
        len(control_train) == EXPECTED["control_train_clips"], len(control_train))
require("control training clips are unique",
        len(set(control_train)) == len(control_train), f"{len(set(control_train)):,}")
require("control training clips exclude the control validation and test clips",
        not (set(control_train) & set(context.control_clip_ids("validation")))
        and not (set(control_train) & set(context.control_clip_ids("test"))),
        "disjoint")

control_standardisers: Dict[str, Any] = {}
for branch in L.BRANCHES:
    started = time.time()

    def progress(done: int, total: int, branch=branch, started=started) -> None:
        elapsed = time.time() - started
        rate = done / max(elapsed, 1e-6)
        print(f"    [{done:,}/{total:,}] {elapsed / 60:.1f} min | {rate:.1f} clips/s | "
              f"eta {(total - done) / max(rate, 1e-6) / 60:.1f} min", flush=True)

    frames, total_sum, total_sumsq, streamed = context.stream_sufficient_statistics(
        control_train, branch, workers=STREAM_WORKERS, progress=progress)

    standardiser = L.standardiser_from_totals(
        branch, frames, total_sum, total_sumsq, scope="control",
        scope_kind="control",
        fitted_on={"kind": "signer_overlap_control", "clips": int(streamed),
                   "frames": int(frames),
                   "source": "streamed over the control-training clips only",
                   "why_not_signer_statistics": "the control shares signers between "
                                                "train and test, so per-signer "
                                                "aggregates contain its test clips"})
    control_standardisers[branch] = standardiser
    context.register_control_standardiser(standardiser)
    require(f"{branch} control standardiser is tagged as control scope",
            standardiser.scope_kind == "control" and standardiser.fold is None,
            "scope_kind=control")
    summary = standardiser.summary()
    print(f"  {branch:<8} {streamed:,} clips, {frames:,} frames in "
          f"{(time.time() - started) / 60:.1f} min | "
          f"mean {summary['mean_min']:.3f}..{summary['mean_max']:.3f} | "
          f"std {summary['std_min']:.3f}..{summary['std_max']:.3f} | "
          f"digest {summary['digest']}")

    require(f"{branch} control pass streamed all 4,500 clips",
            streamed == EXPECTED["control_train_clips"], streamed)
    require(f"{branch} control frame total matches the manifest",
            frames == control_expected_frames, f"{frames:,}")
    require(f"{branch} control standardiser keeps the presence masks binary",
            np.all(standardiser.mean[slice(*EXPECTED["mask_slice"][branch])] == 0.0)
            and np.all(standardiser.std[slice(*EXPECTED["mask_slice"][branch])] == 1.0),
            "mean 0 / std 1")
    require(f"{branch} control standardiser is finite and positive-scaled",
            bool(np.isfinite(standardiser.mean).all()
                 and np.isfinite(standardiser.std).all()
                 and (standardiser.std > 0).all()), "clean")

# The control saw fewer frames than any primary fold standardiser would from all
# 16 signers, and it must differ from them: if it matched a fold standardiser the
# streaming pass would not be doing anything.
for branch in L.BRANCHES:
    control_digest = control_standardisers[branch].digest()
    fold_digests = {fold_standardisers[branch][k].digest()
                    for k in fold_standardisers[branch]}
    require(f"{branch} control standardiser differs from every fold standardiser",
            control_digest not in fold_digests, control_digest)
    require(f"{branch} control used fewer frames than the whole dataset",
            control_standardisers[branch].n_frames < EXPECTED["total_frames"],
            f"{control_standardisers[branch].n_frames:,} of {EXPECTED['total_frames']:,}")

# Same free invariant as the folds: the non-face tail is shared, so it must match.
control_mean_diff = float(np.abs(
    control_standardisers["full"].mean[EXPECTED["face_end"]["full"]:]
    - control_standardisers["reduced"].mean[EXPECTED["face_end"]["reduced"]:]).max())
control_std_diff = float(np.abs(
    control_standardisers["full"].std[EXPECTED["face_end"]["full"]:]
    - control_standardisers["reduced"].std[EXPECTED["face_end"]["reduced"]:]).max())
require("control standardiser tail means identical across branches",
        control_mean_diff == 0.0, f"max {control_mean_diff:.2e}")
require("control standardiser tail stds identical across branches",
        control_std_diff == 0.0, f"max {control_std_diff:.2e}")

for branch in L.BRANCHES:
    path = L.save_standardisers(
        CONTROL_STD_PATH[branch], {"control": control_standardisers[branch]},
        extra={"scope": "signer_overlap_control",
               "fitted_on": "the 4,500 control-training clips only"})
    reloaded = L.load_standardisers(path)
    print(f"wrote {path.name}  {path.stat().st_size:,} bytes")
    require(f"{branch} control standardiser round-trips from disk",
            reloaded["control"].digest() == control_standardisers[branch].digest(),
            reloaded["control"].digest())
    require(f"{branch} control standardiser keeps the control scope on reload",
            reloaded["control"].scope_kind == "control"
            and reloaded["control"].fold is None, "scope_kind=control, fold=None")

In [ ]:
# ============================================================
# CELL 9 — Neural path: reduction, standardisation, padding, masks, determinism
# ============================================================

print("UNIFORM REDUCTION POLICY")
print(f"{'T':>6}  {'kept':>5}  {'first':>5}  {'last':>5}  increasing  policy")
reduction_ok = True
for frames in (32, 100, 255, 256, 257, 300, 402, 1000):
    indices = L.uniform_indices(frames)
    increasing = bool(np.all(np.diff(indices) > 0)) if len(indices) > 1 else True
    policy = "every frame" if frames <= EXPECTED["max_frames"] else "uniform 256"
    print(f"{frames:>6}  {len(indices):>5}  {indices[0]:>5}  {indices[-1]:>5}  "
          f"{str(increasing):>10}  {policy}")
    expected_kept = min(frames, EXPECTED["max_frames"])
    if not (len(indices) == expected_kept and indices[0] == 0
            and indices[-1] == frames - 1 and increasing):
        reduction_ok = False
require("uniform reduction keeps 256 strictly increasing indices spanning the clip",
        reduction_ok, "first and last frame always retained")
require("clips at or below 256 frames keep every original frame",
        np.array_equal(L.uniform_indices(256), np.arange(256))
        and np.array_equal(L.uniform_indices(32), np.arange(32)), "identity")
require("reduction consults no random state",
        np.array_equal(L.uniform_indices(402), L.uniform_indices(402)), "pure function")

# ---- the real long clips ---------------------------------------------------
long_ids = context.manifest.loc[context.manifest["original_frames"] > EXPECTED["max_frames"],
                                "clip_id"].astype(str).tolist()
short_ids = context.manifest.loc[context.manifest["original_frames"] <= EXPECTED["max_frames"],
                                 "clip_id"].astype(str).tolist()
print(f"\nreal clips above {EXPECTED['max_frames']} frames: {len(long_ids)} "
      f"({100 * len(long_ids) / len(context.manifest):.1f}%)")

probe_std = {branch: context.fold_standardiser(0, branch) for branch in L.BRANCHES}
probe_long = sorted(long_ids)[:3] + [str(context.manifest.set_index("clip_id")
                                        ["original_frames"].idxmax())]
probe_short = sorted(short_ids)[:3] + [str(context.manifest.set_index("clip_id")
                                          ["original_frames"].idxmin())]

rows_ok, subsample_ok, mask_ok, precision_ok, tail_ok = True, True, True, True, True
worst_precision = 0.0
for clip_id in sorted(dict.fromkeys(probe_long + probe_short)):
    frames = context.frames_for(clip_id)
    indices = L.uniform_indices(frames)
    standardised = {}
    for branch in L.BRANCHES:
        raw = context.load_raw_sequence(clip_id, branch)
        sequence = context.neural_sequence(clip_id, branch, probe_std[branch])
        standardised[branch] = sequence

        if sequence.shape != (len(indices), EXPECTED["widths"][branch]):
            rows_ok = False
        # the retained rows must be the original rows at those indices
        expected_rows = raw[indices]
        reference = probe_std[branch].apply_float64(expected_rows)
        difference = float(np.abs(sequence.astype(np.float64) - reference).max())
        worst_precision = max(worst_precision, difference)
        if difference > 1e-4:
            precision_ok = False
        start, end = EXPECTED["mask_slice"][branch]
        if not np.array_equal(sequence[:, start:end], expected_rows[:, start:end]):
            mask_ok = False
        if not np.isin(sequence[:, start:end], (0.0, 1.0)).all():
            mask_ok = False
        if sequence.dtype != np.float32:
            rows_ok = False
    if frames > EXPECTED["max_frames"] and standardised["full"].shape[0] != EXPECTED["max_frames"]:
        subsample_ok = False
    if frames <= EXPECTED["max_frames"] and standardised["full"].shape[0] != frames:
        subsample_ok = False
    # standardised non-face columns must still agree across branches
    if not np.array_equal(standardised["full"][:, EXPECTED["face_end"]["full"]:],
                          standardised["reduced"][:, EXPECTED["face_end"]["reduced"]:]):
        tail_ok = False

require("neural sequences have the expected shape and dtype", rows_ok, "float32 (T', D)")
require("clips above 256 frames are reduced, clips at or below are not", subsample_ok,
        f"{len(probe_long)} long, {len(probe_short)} short")
require("presence masks pass through standardisation unchanged and binary", mask_ok, "clean")
require("float32 path matches the float64 reference", precision_ok,
        f"max diff {worst_precision:.2e}")
require("standardised non-face columns identical across branches", tail_ok,
        "355 columns")

# ---- dynamic padding and the temporal mask --------------------------------
print("\nDYNAMIC BATCH PADDING")
batch_ids = sorted(context.clip_ids(0, "test"))
mixed = [str(context.manifest.set_index("clip_id")["original_frames"].idxmin())]
mixed += sorted(long_ids)[:1]
mixed += batch_ids[:2]
mixed = sorted(dict.fromkeys(mixed))

for branch in L.BRANCHES:
    batch = context._make_batch(mixed, branch, probe_std[branch])
    lengths = batch["lengths"]
    print(f"  {branch:<8} lengths {lengths.tolist()} -> padded to "
          f"{batch['sequence'].shape[1]} (max {int(lengths.max())}), "
          f"shape {batch['sequence'].shape}")
    require(f"{branch}: batch padded to the batch maximum, not to 256",
            batch["sequence"].shape[1] == int(lengths.max()),
            f"{batch['sequence'].shape[1]} vs {int(lengths.max())}")
    require(f"{branch}: temporal mask sums to the true lengths",
            int(batch["temporal_mask"].sum()) == int(lengths.sum()),
            f"{int(batch['temporal_mask'].sum()):,}")
    padded_region = batch["sequence"][~batch["temporal_mask"]]
    require(f"{branch}: padded positions are exactly 0.0",
            padded_region.size == 0 or float(np.abs(padded_region).max()) == 0.0,
            f"{padded_region.shape[0]:,} padded frames")
    real_region_rows = batch["temporal_mask"].sum(axis=1)
    require(f"{branch}: every row's real region equals its length",
            np.array_equal(real_region_rows, lengths), "row by row")
    require(f"{branch}: labels and clip ids align with the manifest",
            batch["label_id"].tolist() == [context.label_id_for(c) for c in mixed]
            and batch["clip_id"] == list(mixed), "aligned")
    require(f"{branch}: signer ids carried for per-signer reporting",
            batch["signer_id"] == [context.signer_for(c) for c in mixed], "carried")

# A batch whose members are all the same length must not be padded at all.
equal_ids = (context.manifest[context.manifest["original_frames"]
                              == context.manifest["original_frames"].mode()[0]]
             ["clip_id"].astype(str).sort_values().tolist()[:3])
if len(equal_ids) >= 2:
    equal_batch = context._make_batch(equal_ids, "reduced", probe_std["reduced"])
    require("an equal-length batch is not padded",
            bool(equal_batch["temporal_mask"].all()),
            f"{len(equal_ids)} clips at {int(equal_batch['lengths'][0])} frames")

# ---- determinism -----------------------------------------------------------
print("\nDETERMINISM")
first = context._make_batch(mixed, "full", probe_std["full"])
second = context._make_batch(mixed, "full", probe_std["full"])
require("repeated batches are bit-identical",
        first["sequence"].tobytes() == second["sequence"].tobytes()
        and np.array_equal(first["temporal_mask"], second["temporal_mask"]),
        L.array_digest(first["sequence"]))
require("epoch order is reproducible for a given seed and epoch",
        L.epoch_order(batch_ids, RANDOM_SEED, 0) == L.epoch_order(batch_ids, RANDOM_SEED, 0),
        "same (seed, epoch) -> same order")
require("epoch order changes between epochs",
        L.epoch_order(batch_ids, RANDOM_SEED, 0) != L.epoch_order(batch_ids, RANDOM_SEED, 1),
        "epoch 0 != epoch 1")
require("epoch order is a permutation of the split",
        sorted(L.epoch_order(batch_ids, RANDOM_SEED, 3)) == sorted(batch_ids),
        f"{len(batch_ids):,} clips")
require("the private iterator covers its clips exactly once",
        sorted(c for b in context._iterate_batches(batch_ids[:20], "reduced",
                                                   probe_std["reduced"], batch_size=7)
               for c in b["clip_id"]) == sorted(batch_ids[:20]),
        f"{len(batch_ids[:20])} clips in bounded batches")

# ---- the Random Forest path is untouched ----------------------------------
print("\nRANDOM FOREST PATH (original frames, no reduction, no padding)")
rf_ok = True
for clip_id in sorted(dict.fromkeys(probe_long + probe_short)):
    for branch in L.BRANCHES:
        raw = context.load_raw_sequence(clip_id, branch)
        if raw.shape[0] != context.frames_for(clip_id) \
                or raw.shape[1] != EXPECTED["widths"][branch]:
            rf_ok = False
require("the RF path returns every original frame at full width", rf_ok,
        "no 256-frame reduction, no padding")
untouched = True
for clip_id in sorted(dict.fromkeys(probe_long + probe_short))[:6]:
    for branch in L.BRANCHES:
        on_disk = np.load(context.path_for(clip_id, branch), allow_pickle=False)
        if np.ascontiguousarray(on_disk, dtype=np.float32).tobytes() \
                != context.load_raw_sequence(clip_id, branch).tobytes():
            untouched = False
require("the RF path returns the file on disk unmodified", untouched,
        "no z-scoring applied; derived-feature scaling belongs to notebook 9 and "
        "must be fitted on the training signers there")

# ---- the padding warning, restated where it is used ----------------------
print("\nREQUIREMENT CARRIED TO NOTEBOOKS 11 AND 12")
print("  sequence + lengths must go through")
print("    torch.nn.utils.rnn.pack_padded_sequence(..., enforce_sorted=False)")
print("  or a demonstrably equivalent masking implementation. A bidirectional")
print("  recurrence fed the padded tensor directly starts its backward pass on")
print("  padding and carries that state into the real frames. Pooling over time")
print("  must divide by the true lengths, never by the padded length.")
require("lengths returned for packing", "lengths" in first, "int64 (B,)")
require("temporal mask returned for masking", "temporal_mask" in first, "bool (B, Lmax)")

# ---- the scope-safe API notebooks 10 to 12 must use ------------------------
# Pairing fold 0's clips with fold 1's standardiser is preprocessing leakage that
# no later check would notice. make_fold_batch and make_control_batch fetch the
# right standardiser themselves and refuse a mismatched one.
print("\nSCOPE-SAFE BATCHING API")
subset = sorted(context.clip_ids(0, "test"))[:4]
control_subset = sorted(context.control_clip_ids("test"))[:4]

for branch in L.BRANCHES:
    fold_batch = context.make_fold_batch(0, "test", branch, subset)
    require(f"{branch}: make_fold_batch builds the fold's own batch",
            fold_batch["clip_id"] == subset
            and fold_batch["scope"] == f"fold:0:test:{branch}",
            fold_batch["scope"])
    control_batch = context.make_control_batch("test", branch, control_subset)
    require(f"{branch}: make_control_batch builds the control batch",
            control_batch["clip_id"] == control_subset
            and control_batch["scope"] == f"control:test:{branch}",
            control_batch["scope"])
    require(f"{branch}: the safe API uses the canonical fold standardiser",
            fold_batch["sequence"].tobytes()
            == context._make_batch(subset, branch,
                                   context.fold_standardiser(0, branch))["sequence"].tobytes(),
            "identical bytes to the fold's own standardiser")
    require(f"{branch}: the safe control API uses the canonical control standardiser",
            control_batch["sequence"].tobytes()
            == context._make_batch(control_subset, branch,
                                   context.control_standardiser(branch))["sequence"].tobytes(),
            "identical bytes to the control standardiser")

require("iterate_fold_batches covers a requested subset exactly once",
        sorted(c for b in context.iterate_fold_batches(0, "test", "reduced",
                                                       batch_size=3, clip_ids=subset)
               for c in b["clip_id"]) == subset, f"{len(subset)} clips")
require("iterate_control_batches covers a requested subset exactly once",
        sorted(c for b in context.iterate_control_batches("test", "reduced",
                                                          batch_size=3,
                                                          clip_ids=control_subset)
               for c in b["clip_id"]) == control_subset, f"{len(control_subset)} clips")
rf_stream = list(context.fold_raw_sequences(0, "test", "full", clip_ids=subset))
require("fold_raw_sequences yields original unscaled frames",
        all(item["sequence"].shape == (context.frames_for(item["clip_id"]),
                                       EXPECTED["widths"]["full"])
            for item in rf_stream) and len(rf_stream) == len(subset),
        f"{len(rf_stream)} clips, no reduction or padding")
require("control_raw_sequences yields original unscaled frames",
        all(item["sequence"].shape[1] == EXPECTED["widths"]["reduced"]
            for item in context.control_raw_sequences("test", "reduced",
                                                      clip_ids=control_subset)),
        f"{len(control_subset)} clips")

print("\nSCOPE VIOLATIONS (each must be refused, never quietly rescaled)")
ad_hoc = L.standardiser_from_totals("reduced", 100, np.zeros(631), np.ones(631),
                                    scope="probe")
scope_probes = [
    # Standardiser / data mismatches, checked through the guard the safe API uses.
    ("fold 0 data cannot use the fold 1 standardiser",
     lambda: context.require_standardiser_scope(
         context.fold_standardiser(1, "full"), "full", "fold", 0), L.ScopeError),
    ("full data cannot use the reduced standardiser",
     lambda: context.require_standardiser_scope(
         context.fold_standardiser(0, "reduced"), "full", "fold", 0), L.ScopeError),
    ("reduced data cannot use the full standardiser",
     lambda: context.require_standardiser_scope(
         context.fold_standardiser(0, "full"), "reduced", "fold", 0), L.ScopeError),
    ("control data cannot use a fold standardiser",
     lambda: context.require_standardiser_scope(
         context.fold_standardiser(0, "full"), "full", "control"), L.ScopeError),
    ("fold data cannot use the control standardiser",
     lambda: context.require_standardiser_scope(
         context.control_standardiser("full"), "full", "fold", 0), L.ScopeError),
    ("an ad-hoc standardiser is refused",
     lambda: context.require_standardiser_scope(ad_hoc, "reduced", "fold", 0),
     L.ScopeError),
    ("a non-Standardiser object is refused",
     lambda: context.require_standardiser_scope("not a standardiser", "full", "fold", 0),
     L.ScopeError),
    # Clip-id scope violations, through the public API.
    ("clip ids from another split are refused",
     lambda: context.make_fold_batch(0, "test", "reduced",
                                     sorted(context.clip_ids(0, "train"))[:2]),
     L.ScopeError),
    ("clip ids from another fold's test split are refused",
     lambda: context.make_fold_batch(0, "test", "reduced",
                                     sorted(context.clip_ids(1, "test"))[:2]),
     L.ScopeError),
    ("control-training clip ids are refused by the fold API",
     lambda: context.make_fold_batch(
         0, "test", "reduced",
         sorted(set(context.control_clip_ids("train"))
                - set(context.clip_ids(0, "test")))[:2]), L.ScopeError),
    ("fold clip ids outside the control split are refused by the control API",
     lambda: context.make_control_batch(
         "test", "reduced",
         sorted(set(context.manifest["clip_id"])
                - set(context.control_clip_ids("test")))[:2]), L.ScopeError),
    ("duplicate clip ids are refused",
     lambda: context.make_fold_batch(0, "test", "reduced", [subset[0], subset[0]]),
     L.ScopeError),
    ("a fold standardiser cannot be registered as the control standardiser",
     lambda: context.register_control_standardiser(context.fold_standardiser(0, "full")),
     L.ScopeError),
    # A whole split as one batch is refused outright.
    ("make_fold_batch refuses clip_ids=None",
     lambda: context.make_fold_batch(0, "test", "reduced", None), ValueError),
    ("make_control_batch refuses clip_ids=None",
     lambda: context.make_control_batch("test", "reduced", None), ValueError),
    ("make_fold_batch refuses a missing clip_ids argument",
     lambda: context.make_fold_batch(0, "test", "reduced"), TypeError),
    ("make_control_batch refuses a missing clip_ids argument",
     lambda: context.make_control_batch("test", "reduced"), TypeError),
    ("a zero batch size is refused",
     lambda: next(context.iterate_fold_batches(0, "test", "reduced", batch_size=0)),
     ValueError),
]
for name, call, expected in scope_probes:
    try:
        call()
        require(name, False, "accepted")
    except expected as error:
        record(name, True, str(error).split(".")[0][:58])

# ---- the safe API must offer no substitution route -------------------------
print("\nNON-BYPASSABILITY OF THE SAFE API")
import inspect

for name in ("make_fold_batch", "make_control_batch",
             "iterate_fold_batches", "iterate_control_batches",
             "fold_raw_sequences", "control_raw_sequences"):
    parameters = inspect.signature(getattr(context, name)).parameters
    require(f"{name} exposes no standardiser argument",
            "standardiser" not in parameters, ", ".join(parameters))

for name in ("make_fold_batch", "make_control_batch"):
    parameters = inspect.signature(getattr(context, name)).parameters
    require(f"{name} has no default for clip_ids",
            parameters["clip_ids"].default is inspect.Parameter.empty,
            "explicit clip ids required")

require("the unguarded batch helpers are private",
        not hasattr(context, "make_batch") and not hasattr(context, "iterate_batches")
        and hasattr(context, "_make_batch") and hasattr(context, "_iterate_batches"),
        "_make_batch / _iterate_batches")
public_batch_api = sorted(name for name in dir(context)
                          if not name.startswith("_") and "batch" in name)
require("the only public batch entry points are the scope-safe ones",
        public_batch_api == ["iterate_control_batches", "iterate_fold_batches",
                             "make_control_batch", "make_fold_batch"],
        ", ".join(public_batch_api))

# ---- bounded mini-batch coverage of a whole split --------------------------
# The iterators are the only route to a complete split, and they must cover it
# exactly once while never holding more than batch_size clips in memory.
print("\nBOUNDED MINI-BATCH COVERAGE OF A WHOLE SPLIT")
sweep_fold = min(range(EXPECTED["folds"]), key=lambda f: len(context.clip_ids(f, "test")))
sweep_ids = context.clip_ids(sweep_fold, "test")
# Sized so the sweep always spans several batches, whatever the split holds.
SWEEP_BATCH = min(32, max(2, len(sweep_ids) // 4))
seen, batch_sizes, batch_bytes, padded_lengths = [], [], [], []
for batch in context.iterate_fold_batches(sweep_fold, "test", "reduced",
                                          batch_size=SWEEP_BATCH):
    seen.extend(batch["clip_id"])
    batch_sizes.append(len(batch["clip_id"]))
    batch_bytes.append(int(batch["sequence"].nbytes))
    padded_lengths.append(int(batch["padded_length"]))
ceiling = SWEEP_BATCH * EXPECTED["max_frames"] * EXPECTED["widths"]["reduced"] * 4
print(f"  fold {sweep_fold} test, reduced branch: {len(sweep_ids):,} clips in "
      f"{len(batch_sizes)} batches of at most {SWEEP_BATCH}")
print(f"  largest batch {max(batch_bytes) / 1e6:.1f} MB (ceiling "
      f"{ceiling / 1e6:.1f} MB) | padded lengths {min(padded_lengths)}-"
      f"{max(padded_lengths)}")
require("iterate_fold_batches covers a whole split exactly once",
        sorted(seen) == sorted(sweep_ids) and len(seen) == len(set(seen)),
        f"{len(seen):,} of {len(sweep_ids):,} clips")
require("no mini-batch exceeds the requested batch size",
        max(batch_sizes) <= SWEEP_BATCH, f"max {max(batch_sizes)}")
require("every mini-batch stays inside the memory ceiling",
        max(batch_bytes) <= ceiling, f"{max(batch_bytes) / 1e6:.1f} MB")
require("no mini-batch is padded beyond MAX_FRAMES",
        max(padded_lengths) <= EXPECTED["max_frames"], max(padded_lengths))

control_seen, control_sizes = [], []
for batch in context.iterate_control_batches("test", "reduced", batch_size=SWEEP_BATCH):
    control_seen.extend(batch["clip_id"])
    control_sizes.append(len(batch["clip_id"]))
require("iterate_control_batches covers a whole split exactly once",
        sorted(control_seen) == sorted(context.control_clip_ids("test"))
        and len(control_seen) == len(set(control_seen)),
        f"{len(control_seen):,} clips in {len(control_sizes)} batches")
require("shuffled iteration covers the same split exactly once",
        sorted(c for b in context.iterate_fold_batches(
            sweep_fold, "test", "reduced", batch_size=SWEEP_BATCH, shuffle=True,
            seed=RANDOM_SEED, epoch=1) for c in b["clip_id"]) == sorted(sweep_ids),
        "order changes, membership does not")

# ---- the torch bridge ------------------------------------------------------
print("\nTORCH BRIDGE (lengths must stay on the CPU for pack_padded_sequence)")
try:
    import torch
except ImportError:
    skip("torch bridge keeps lengths on the CPU", "torch not installed")
    skip("pack_padded_sequence accepts the returned lengths", "torch not installed")
else:
    bridge_batch = context.make_fold_batch(0, "test", "reduced", subset)
    on_cpu = L.to_torch(bridge_batch)
    require("to_torch returns lengths as int64 on the CPU",
            on_cpu["lengths"].device.type == "cpu"
            and on_cpu["lengths"].dtype == torch.int64,
            f"{on_cpu['lengths'].dtype} on {on_cpu['lengths'].device}")
    require("to_torch declares lengths as device exempt",
            on_cpu["device_exempt"] == ["lengths"], on_cpu["device_exempt"])
    require("to_torch preserves the batch scope", on_cpu["scope"] == bridge_batch["scope"],
            on_cpu["scope"])

    # Probe with a real non-CPU device. On a CPU-only session the meta device is
    # enough to prove the move happens and that lengths is excluded from it.
    probe_device: Optional[str] = None
    if torch.cuda.is_available():
        probe_device = "cuda"
    else:
        try:
            torch.zeros(1).to("meta")
            probe_device = "meta"
        except Exception:
            probe_device = None

    if probe_device is None:
        skip("lengths stay on the CPU when a device is requested",
             "no non-CPU device available to probe")
    else:
        moved = L.to_torch(bridge_batch, device=probe_device)
        require(f"sequence moves to the requested device ({probe_device})",
                moved["sequence"].device.type == probe_device,
                moved["sequence"].device)
        require(f"temporal_mask moves to the requested device ({probe_device})",
                moved["temporal_mask"].device.type == probe_device,
                moved["temporal_mask"].device)
        require(f"label_id moves to the requested device ({probe_device})",
                moved["label_id"].device.type == probe_device, moved["label_id"].device)
        require(f"lengths stay on the CPU when {probe_device} is requested",
                moved["lengths"].device.type == "cpu", moved["lengths"].device)

    packed = torch.nn.utils.rnn.pack_padded_sequence(
        on_cpu["sequence"], on_cpu["lengths"], batch_first=True, enforce_sorted=False)
    unpacked, recovered_lengths = torch.nn.utils.rnn.pad_packed_sequence(
        packed, batch_first=True)
    require("pack_padded_sequence accepts the returned lengths",
            bool(torch.equal(recovered_lengths, on_cpu["lengths"])),
            f"{int(on_cpu['lengths'].sum())} real frames packed")
    real = on_cpu["temporal_mask"][:, :unpacked.shape[1]]
    require("packing then unpacking recovers every real frame",
            bool(torch.equal(unpacked[real], on_cpu["sequence"][:, :unpacked.shape[1]][real])),
            "no padding entered the packed sequence")

In [ ]:
# ============================================================
# CELL 10 — Freeze the contract, hash the module, then write the report
# ============================================================
# Ordering matters here. The scientific contract is hashed first, the published
# module is hashed and re-verified from disk, the manifest is written and read
# back, and only after every one of those checks has run is the validation report
# written. The saved report therefore describes the finished state rather than a
# snapshot taken part-way through.

STAGE = "contract"
early_failures = failures()
if early_failures:
    for entry in early_failures:
        print(f"  FAIL  {entry['check']}: {entry['detail']}")

# ---- A. the stable scientific contract ------------------------------------
# Policy, boundaries and verified provenance only. No timestamp, no session path,
# no runtime metadata: rerunning this loader tomorrow must reproduce the same id.
CONTRACT = L.loader_contract(extra={
    "notebook": "8-shared-sequence-loader",
    "module_filename": MODULE_PATH.name,
    "verified_provenance": {
        "experiment_id": REQUIRED_EXPERIMENT_ID,
        "manifest_version": REQUIRED_MANIFEST_VERSION,
        "pipeline_id": REQUIRED_PIPELINE_ID,
        "clip_table_hash": REQUIRED_CLIP_TABLE_HASH,
        "successful_clip_id_hash": REQUIRED_SUCCESSFUL_CLIP_ID_HASH,
        "notebook7_hashes": dict(REQUIRED_HASHES),
    },
    "counts": {"clips": EXPECTED["clips"], "labels": EXPECTED["labels"],
               "signers": EXPECTED["signers"], "folds": EXPECTED["folds"],
               "total_frames": EXPECTED["total_frames"],
               "clips_over_max_frames": int(
                   (context.manifest["original_frames"] > EXPECTED["max_frames"]).sum()),
               "fold_standardisers": EXPECTED["folds"] * len(L.BRANCHES),
               "control_standardisers": len(L.BRANCHES)},
    "downstream_obligations": {
        "before_import": "hash bsl_sequence_loader.py and compare it with "
                         "loader_manifest.json module.sha256 BEFORE importing it; an "
                         "edited data-access layer invalidates every comparison built "
                         "on it",
        "notebook_9": "Random Forest feature engineering over ORIGINAL frames only, "
                      "via fold_raw_sequences(fold, split, branch). Any variance "
                      "filter, selection or PCA is a derived-feature transformation "
                      "in a different space from this loader's frame-level "
                      "standardisers and must be fitted on the 12 training signers of "
                      "each fold inside that notebook. Report that the full branch "
                      "yields roughly 2.8x more features than the reduced branch, so "
                      "an RF difference partly reflects dimensionality rather than "
                      "facial information.",
        "notebook_10": "Random Forest, 8 folds x 2 branches plus one control run per "
                       "branch. Read these folds; never regenerate a split.",
        "notebook_11_12": "BiLSTM and BiGRU, 8 folds x 2 branches plus one control run "
                          "per branch. Use the scope-safe API, feed sequence and "
                          "lengths through pack_padded_sequence(..., "
                          "enforce_sorted=False) or a demonstrably equivalent masking "
                          "implementation, and keep lengths on the CPU. Freeze the "
                          "training procedure before any test evaluation and keep it "
                          "identical between the full and reduced branches of the same "
                          "model. Report exact parameter counts: the recurrent core is "
                          "matched across branches but the input projection differs, "
                          "so totals are not identical.",
        "notebook_13": "Primary result is the pooled 6,000 out-of-fold predictions. "
                       "Variability as mean and std across the 8 folds. Primary "
                       "inference is the pre-specified exhaustive paired sign-flip "
                       "test over the 8 fold-level differences, interpreted cautiously "
                       "because cross-validation training sets overlap.",
    },
})
L.assert_stable_contract(CONTRACT)
CONTRACT_ID = L.contract_id(CONTRACT)

# ---- B. the fitted scalings, hashed separately from the policy -------------
STANDARDISER_DIGESTS = {
    "folds": {branch: {key: fold_standardisers[branch][key].digest()
                       for key in sorted(fold_standardisers[branch])}
              for branch in L.BRANCHES},
    "control": {branch: control_standardisers[branch].digest() for branch in L.BRANCHES},
}
STANDARDISER_SET_ID = L.standardiser_set_id(STANDARDISER_DIGESTS)

print("LOADER CONTRACT")
print(f"  loader contract id   : {CONTRACT_ID}   (policy and provenance, stable)")
print(f"  standardiser set id  : {STANDARDISER_SET_ID}   (the fitted scalings)")
require("the contract id excludes run-specific content",
        not (set(_all_keys(CONTRACT)) & set(L.VOLATILE_CONTRACT_KEYS)),
        "no timestamp, session path or runtime metadata")
require("the contract id is reproducible from a serialised round trip",
        L.contract_id(json.loads(json.dumps(CONTRACT, default=str))) == CONTRACT_ID,
        CONTRACT_ID)

# ---- C. hash the published module -----------------------------------------
STAGE = "module hash"
MODULE_SHA256 = L.module_sha256(MODULE_PATH)
MODULE_BYTES = MODULE_PATH.stat().st_size
print("\nPUBLISHED MODULE")
print(f"  file                 : {MODULE_PATH.name}")
print(f"  sha256               : {MODULE_SHA256}")
print(f"  size                 : {MODULE_BYTES:,} bytes, "
      f"{len(MODULE_SOURCE.splitlines()):,} lines")

require("the published module is byte-identical to what was tested",
        MODULE_PATH.read_text(encoding="utf-8") == MODULE_SOURCE,
        f"{MODULE_BYTES:,} bytes")
require("the module hash recomputes from the file on disk",
        L.module_sha256(MODULE_PATH) == MODULE_SHA256, MODULE_SHA256[:16] + "...")
require("verify_module accepts the recorded hash",
        L.verify_module(MODULE_SHA256, MODULE_PATH) == MODULE_SHA256, "accepted")
try:
    L.verify_module("0" * 64, MODULE_PATH)
    require("verify_module rejects a wrong hash", False, "accepted")
except L.ProvenanceError:
    record("verify_module rejects a wrong hash", True, "ProvenanceError")

MODULE_BLOCK = {"filename": MODULE_PATH.name, "sha256": MODULE_SHA256,
                "sha256_short": MODULE_SHA256[:16], "bytes": MODULE_BYTES,
                "lines": len(MODULE_SOURCE.splitlines()),
                "loader_version": LOADER_VERSION,
                "verify_before_import":
                    "hashlib.sha256(open(path, 'rb').read()).hexdigest()"}

# ---- D. the manifest -------------------------------------------------------
STAGE = "manifest"
RUN_METADATA = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "environment": ENVIRONMENT,
    "resolved_paths_this_session": context.provenance["resolved"],
    "csv_read_strategy": context.provenance["csv_read_strategy"],
    "note": "recorded for traceability and deliberately excluded from "
            "loader_contract_id, which must not change between reruns",
}
MANIFEST = {
    "loader_contract_id": CONTRACT_ID,
    "loader_contract": CONTRACT,
    "standardiser_set_id": STANDARDISER_SET_ID,
    "standardiser_digests": STANDARDISER_DIGESTS,
    "module": MODULE_BLOCK,
    "artefacts": {
        "fold_standardisers": {b: FOLD_STD_PATH[b].name for b in L.BRANCHES},
        "control_standardisers": {b: CONTROL_STD_PATH[b].name for b in L.BRANCHES},
        "validation_report": [REPORT_JSON.name, REPORT_CSV.name],
    },
    "run_metadata": RUN_METADATA,
}


def atomic_write(path: Path, writer) -> None:
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        writer(handle)
    os.replace(temporary, path)


atomic_write(LOADER_MANIFEST, lambda h: h.write(
    json.dumps(MANIFEST, indent=2, sort_keys=True, default=str).encode()))

# Recompute both ids from what is actually on disk. Hashing what is in memory
# would only prove the computation was self-consistent.
reloaded_manifest = json.loads(LOADER_MANIFEST.read_text(encoding="utf-8"))
require("loader contract id reproduces from the manifest on disk",
        L.contract_id(reloaded_manifest["loader_contract"]) == CONTRACT_ID
        == reloaded_manifest["loader_contract_id"], CONTRACT_ID)
require("standardiser set id reproduces from the manifest on disk",
        L.standardiser_set_id(reloaded_manifest["standardiser_digests"])
        == STANDARDISER_SET_ID == reloaded_manifest["standardiser_set_id"],
        STANDARDISER_SET_ID)
require("the manifest records the module hash",
        reloaded_manifest["module"]["sha256"] == MODULE_SHA256, "module.sha256")
require("run metadata is stored outside the hashed contract",
        "run_metadata" in reloaded_manifest
        and "generated_at_utc" not in json.dumps(reloaded_manifest["loader_contract"]),
        "separated")
require("the recorded standardiser digests match the live objects",
        reloaded_manifest["standardiser_digests"] == STANDARDISER_DIGESTS,
        f"{len(L.BRANCHES) * (EXPECTED['folds'] + 1)} scalings")

# ---- E. final gate, then the report --------------------------------------
STAGE = "report"
FINAL_FAILURES = failures()
SKIPPED = [entry for entry in CHECKS if entry["status"] == "skip"]
VERDICT = "FAIL" if FINAL_FAILURES else "PASS"

report = {
    "notebook": "8-shared-sequence-loader", "loader_version": LOADER_VERSION,
    "verdict": VERDICT,
    "loader_contract_id": CONTRACT_ID,
    "standardiser_set_id": STANDARDISER_SET_ID,
    "experiment_id": REQUIRED_EXPERIMENT_ID,
    "module": MODULE_BLOCK,
    "checks_total": len(CHECKS),
    "checks_passed": len([c for c in CHECKS if c["status"] == "pass"]),
    "checks_failed": len(FINAL_FAILURES),
    "checks_skipped": len(SKIPPED),
    "checks": CHECKS,
    "fold_table": fold_table.to_dict("records"),
    "standardiser_table": standardiser_table.to_dict("records"),
    "control_standardisers": {branch: control_standardisers[branch].summary()
                              for branch in L.BRANCHES},
    "runtime_seconds": round(time.time() - STARTED, 1),
    "run_metadata": RUN_METADATA,
}
atomic_write(REPORT_JSON, lambda h: h.write(
    json.dumps(report, indent=2, default=str).encode()))
atomic_write(REPORT_CSV, lambda h: pd.DataFrame(CHECKS)[
    ["check", "status", "passed", "detail"]].to_csv(h, index=False))
print(f"\nwrote {REPORT_JSON.name} and {REPORT_CSV.name} "
      f"({len(CHECKS)} checks, verdict {VERDICT})")

if FINAL_FAILURES:
    for entry in FINAL_FAILURES:
        print(f"  FAIL  {entry['check']}: {entry['detail']}")
    raise RuntimeError(
        f"{len(FINAL_FAILURES)} checks failed. The validation report was written for "
        "diagnosis, but the loader contract is NOT usable and no downstream notebook "
        "should read these artefacts.")

written = [MODULE_PATH, LOADER_MANIFEST, REPORT_JSON, REPORT_CSV,
           *FOLD_STD_PATH.values(), *CONTROL_STD_PATH.values()]
total_bytes = sum(p.stat().st_size for p in written)

print("=" * 78)
print("NOTEBOOK 8 COMPLETE - LOADER CONTRACT FROZEN")
print("=" * 78)
print(f"  experiment id        : {REQUIRED_EXPERIMENT_ID}")
print(f"  loader version       : {LOADER_VERSION}")
print(f"  loader contract id   : {CONTRACT_ID}")
print(f"  standardiser set id  : {STANDARDISER_SET_ID}")
print(f"  module sha256        : {MODULE_SHA256}")
print(f"  checks               : {report['checks_passed']} passed, "
      f"{report['checks_failed']} failed, {report['checks_skipped']} skipped "
      f"of {len(CHECKS)}")
print(f"  runtime              : {(time.time() - STARTED) / 60:.1f} min")
print(f"  outputs              : {total_bytes / 1024:.0f} KB total, no tensors exported")

print("\n  Files written:")
for path in written:
    print(f"    {path.name:<38} {path.stat().st_size:>10,} bytes")

print("\n  Frozen here: folds, path reconstruction, raw loading, the 256-frame")
print("  policy, uniform reduction, dynamic padding, temporal masks, presence-mask")
print("  handling, training-only fold standardisation, control-only standardisation,")
print("  scope-safe batching with no standardiser override and no whole-split batch,")
print("  CPU lengths, deterministic loader behaviour.")
print("\n  NOT frozen here: optimiser, learning rate, epochs, patience, dropout,")
print("  architecture, Random Forest settings. Those belong to notebooks 10, 11 and")
print("  12 and must be frozen there before any test evaluation. Within a model, the")
print("  full and reduced branches must follow an identical procedure; BiLSTM and")
print("  BiGRU need not match each other.")

print("\n  Downstream notebooks start with:")
print("    import hashlib, json, sys")
print("    root = '<attached notebook 8 output>'")
print("    manifest = json.load(open(f'{root}/loader_manifest.json'))")
print("    digest = hashlib.sha256("
      "open(f'{root}/bsl_sequence_loader.py', 'rb').read()).hexdigest()")
print("    assert digest == manifest['module']['sha256'], 'loader module was modified'")
print("    sys.path.insert(0, root); import bsl_sequence_loader as L")
print(f"    context = L.build_context()        # refuses anything but {REQUIRED_EXPERIMENT_ID}")
print("    # neural path: bounded mini-batches, never a whole split at once")
print("    for batch in context.iterate_fold_batches(fold, 'train', branch,")
print("                                             batch_size=16, shuffle=True,")
print("                                             seed=42, epoch=epoch):")
print("        tensors = L.to_torch(batch, device=device)   # lengths stay on the CPU")
print("    for batch in context.iterate_control_batches('train', branch, batch_size=16):")
print("        ...")
print("    # Random Forest path: original frames, one clip at a time")
print("    for item in context.fold_raw_sequences(fold, 'train', branch): ...")

print("\n" + "=" * 78)
print(f"  Quote loader contract id {CONTRACT_ID}, standardiser set id "
      f"{STANDARDISER_SET_ID}")
print(f"  and experiment id {REQUIRED_EXPERIMENT_ID} in notebooks 9 to 13.")
print("  STOP. Do not build notebook 9 until this real Kaggle log has been reviewed.")
print("=" * 78)